# Radar Financeiro — MVP para um cliente

Notebook de desenvolvimento deliberadamente granular. Recebe um único CD_CLI e produz somente a view temporária vw_radar_financeiro_cliente_mvp, com uma linha e as 80 colunas do contrato vigente.

Regras bloqueantes: no máximo cinco consultas externas, uma consulta por célula, nenhuma escrita externa, nenhuma classificação por fallback e nenhuma correção de cardinalidade com DISTINCT.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao='radar-financeiro-mvp-cliente',
        exibir_configuracao=False,
        ativar_logs=True,
    )
    spark = gerenciador_local.criar_sessao_spark(db2=True)
    print('[RADAR_MVP] Sessão Spark inicializada pelo padrão corporativo.')
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


## Utilitários corporativos

Os gerenciadores existentes são carregados sem criar uma camada alternativa de sessão ou conexão.


In [ ]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


## Configuração central e observabilidade


In [ ]:
%%spark

import calendar
import datetime
import os
import re
import time
from collections import defaultdict
from decimal import Decimal, ROUND_HALF_UP
from datetime import timedelta

from pyspark.sql import Row, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, ShortType,
    StringType, DateType, TimestampType, DecimalType
)
from pyspark.storagelevel import StorageLevel

FONTE_TRAN = 'DB2GFP.TRAN_RLZD_INST_PCT'
FONTE_CICLO = 'DB2GFP.CT_GRDR_FNCO'
FONTE_RENDA = 'DB2DFE.REN_AVLD_PF'
FONTE_PERFIL = 'DB2D1D.DVS_GRDR_FNCO_PF'
VIEW_RESULTADO = 'vw_radar_financeiro_cliente_mvp'
FETCHSIZE = 10_000
QUERY_TIMEOUT_SECONDS = 900

DATA_EXECUCAO = datetime.date.fromisoformat(str(obter_variavel_ambiente('HOJE'))[:10])

def recuar_um_mes_calendario(data):
    total = data.year * 12 + data.month - 2
    ano, mes_zero = divmod(total, 12)
    mes = mes_zero + 1
    return datetime.date(ano, mes, min(data.day, calendar.monthrange(ano, mes)[1]))

DATA_INICIAL_PUBLICO = recuar_um_mes_calendario(DATA_EXECUCAO)
DATA_FINAL_EXCLUSIVA_PUBLICO = DATA_EXECUCAO
DT_MES_EXEA = DATA_EXECUCAO.replace(day=1)

METRICAS = []
INICIO_MVP = time.perf_counter()

def registrar_medicao(tipo, nome, fonte, inicio_iso, fim_iso, duracao, linhas, colunas, status, detalhe='', preparacao=None, materializacao=None):
    METRICAS.append({
        'TIPO': tipo,
        'ETAPA_QUERY': nome,
        'FONTE': fonte,
        'CD_CLI': CD_CLI if 'CD_CLI' in globals() else None,
        'INICIO': inicio_iso,
        'FIM': fim_iso,
        'TEMPO_SEG': float(duracao),
        'TEMPO_PREPARACAO_SEG': None if preparacao is None else float(preparacao),
        'TEMPO_MATERIALIZACAO_SEG': None if materializacao is None else float(materializacao),
        'LINHAS': None if linhas is None else int(linhas),
        'COLUNAS': None if colunas is None else int(colunas),
        'STATUS': status,
        'DETALHE': detalhe,
    })

def medir_transformacao(nome, inicio, linhas=None, colunas=None, status='OK', detalhe=''):
    fim = time.perf_counter()
    duracao = fim - inicio
    fim_wall = datetime.datetime.now()
    inicio_wall = fim_wall - timedelta(seconds=duracao)
    registrar_medicao('TRANSFORMACAO', nome, 'SPARK/LOCAL', inicio_wall.isoformat(), fim_wall.isoformat(), duracao, linhas, colunas, status, detalhe)

conector_db2 = criar_conector_db2_spark(env=dict(os.environ))

print(f'[RADAR_MVP] DATA_EXECUCAO={DATA_EXECUCAO}')
print(f'[RADAR_MVP] JANELA_PUBLICO={DATA_INICIAL_PUBLICO} <= TS_INCL_TRAN < {DATA_FINAL_EXCLUSIVA_PUBLICO}')
print(f'[RADAR_MVP] FETCHSIZE={FETCHSIZE}; QUERY_TIMEOUT_SECONDS={QUERY_TIMEOUT_SECONDS}')


In [ ]:
%%spark

# V23 — GUARD DE ESTADO DA EXECUÇÃO
STATUS_SCHEMA_Q3 = 'NOT_EVALUATED'
STATUS_SCHEMA_Q5 = 'NOT_EVALUATED'
STATUS_GATE_ANULACAO = 'NOT_EVALUATED'
STATUS_TESTES_SINTETICOS = 'NOT_EXECUTED'
STATUS_CONTRATO_FINAL = 'NOT_EVALUATED'
STATUS_VIEW_FINAL = 'NOT_CREATED'
HOMOLOGACAO_FUNCIONAL_V23 = 'N'
PERFORMANCE_Q4_HOMOLOGADA = 'N'
ESTRATEGIA_Q4 = 'FALLBACK_FUNCIONAL_SPARK'

def obter_temp_views_resultado():
    return [
        objeto for objeto in spark.catalog.listTables()
        if objeto.name == VIEW_RESULTADO and bool(objeto.isTemporary)
    ]

def remover_temp_view_resultado():
    temp_views = obter_temp_views_resultado()
    if len(temp_views) > 1:
        raise AssertionError(
            f'Estado ambíguo: {len(temp_views)} temporary views chamadas {VIEW_RESULTADO}.'
        )
    if temp_views:
        if not spark.catalog.dropTempView(VIEW_RESULTADO):
            raise AssertionError(f'Não foi possível remover a temporary view {VIEW_RESULTADO}.')
    restantes = obter_temp_views_resultado()
    if restantes:
        raise AssertionError(f'A temporary view {VIEW_RESULTADO} permaneceu no catálogo.')

remover_temp_view_resultado()
print(f'[V23][GUARD] temporary view anterior ausente: {VIEW_RESULTADO}')
print(f'[V23][GUARD] ESTRATEGIA_Q4={ESTRATEGIA_Q4}; PERFORMANCE_Q4_HOMOLOGADA={PERFORMANCE_Q4_HOMOLOGADA}')


In [ ]:
%%spark

# V23 — CONFIGURAÇÃO DE AUDITORIA

AUDITORIA_ATIVA = True
LIMITE_EXIBICAO = 20
LIMITE_VALORES_UNICOS = 30
AUDITORIA_PLANOS_SPARK = False
LIMITE_CARACTERES_PLANO = 8000
SCHEMA_Q3_SKIPPED = StructType([
    StructField('NR_CPF', DecimalType(11, 0), True),
    StructField('DT_INCL_REN_AVLD', DateType(), True),
    StructField('VL_REN', DecimalType(17, 2), True),
])
SCHEMA_Q5_SKIPPED = StructType([
    StructField('NR_TRAN_INST_PCT', LongType(), True),
    StructField('CD_CLI', IntegerType(), True),
    StructField('DT_TRAN', DateType(), True),
    StructField('CD_NTZ_CTB_TRAN', StringType(), True),
    StructField('CD_CTGR_TRAN_OGNL', IntegerType(), True),
    StructField('CD_TIP_MOE_CRR', StringType(), True),
    StructField('VL_TRAN', DecimalType(15, 2), True),
])

def registrar_auditoria(nome, inicio, linhas=None, colunas=None, status='OK', detalhe=''):
    fim = time.perf_counter()
    duracao = fim - inicio
    fim_wall = datetime.datetime.now()
    inicio_wall = fim_wall - timedelta(seconds=duracao)
    registrar_medicao('AUDITORIA', nome, 'SPARK/LOCAL', inicio_wall.isoformat(), fim_wall.isoformat(), duracao, linhas, colunas, status, detalhe)

def medir_validacao(nome, inicio, linhas=None, colunas=None, status='OK', detalhe=''):
    fim = time.perf_counter()
    duracao = fim - inicio
    fim_wall = datetime.datetime.now()
    inicio_wall = fim_wall - timedelta(seconds=duracao)
    registrar_medicao('VALIDACAO', nome, 'SPARK/LOCAL', inicio_wall.isoformat(), fim_wall.isoformat(), duracao, linhas, colunas, status, detalhe)

def auditar_schema(nome, df):
    print(f'\n[AUDITORIA] SCHEMA | {nome}')
    print('COLUNA | TIPO | NULLABLE')
    for campo in df.schema.fields:
        print(f'{campo.name} | {campo.dataType.simpleString()} | {campo.nullable}')

def comparar_schema_referencia(nome, df, schema_referencia):
    atual = [(f.name, f.dataType.simpleString(), f.nullable) for f in df.schema.fields]
    referencia = [(f.name, f.dataType.simpleString(), f.nullable) for f in schema_referencia.fields]
    status = 'OK' if atual == referencia else 'DIVERGENTE'
    print(f'\n[AUDITORIA] SCHEMA EXECUTADO × SKIPPED | {nome} | STATUS={status}')
    print('ORDEM | EXECUTADO | SKIPPED | STATUS')
    for indice in range(max(len(atual), len(referencia))):
        item_atual = atual[indice] if indice < len(atual) else None
        item_referencia = referencia[indice] if indice < len(referencia) else None
        print(f'{indice + 1} | {item_atual} | {item_referencia} | {"OK" if item_atual == item_referencia else "DIVERGENTE"}')
    return status

def auditar_linhas(nome, df, quantidade):
    limite = LIMITE_EXIBICAO
    print(f'\n[AUDITORIA] {nome} | linhas={quantidade} | colunas={len(df.columns)}')
    if quantidade <= limite:
        df.show(quantidade, truncate=False)
    else:
        df.limit(limite).show(limite, truncate=False)
        print(f'[AUDITORIA] exibição limitada a {limite} linhas.')

def auditar_valores_controlados(nome, df, colunas):
    for coluna in colunas:
        df_aud_valores = df.groupBy(coluna).count()
        qt_distintos = df_aud_valores.count()
        print(f'\n[AUDITORIA] {nome}.{coluna} | distintos={qt_distintos}')
        if qt_distintos <= LIMITE_VALORES_UNICOS:
            df_aud_valores.orderBy(F.col(coluna).asc_nulls_first()).show(LIMITE_VALORES_UNICOS, truncate=False)
        else:
            df_aud_resumo = df.agg(
                F.sum(F.when(F.col(coluna).isNull(), 1).otherwise(0)).alias('NULLS'),
                F.min(F.col(coluna)).alias('MIN'),
                F.max(F.col(coluna)).alias('MAX')
            )
            df_aud_resumo.show(truncate=False)
            df_aud_valores.limit(LIMITE_EXIBICAO).show(LIMITE_EXIBICAO, truncate=False)

def auditar_painel(titulo, pares):
    print(f'\n[AUDITORIA] {titulo}')
    for chave, valor in pares:
        print(f'{chave} = {valor}')


## Parâmetro obrigatório


In [ ]:
%%spark

CD_CLI = None

if CD_CLI is None:
    raise RuntimeError('BLOQUEADO: informe um único CD_CLI inteiro nesta célula.')
if isinstance(CD_CLI, bool):
    raise TypeError('CD_CLI deve ser inteiro, não booleano.')
texto_cd_cli = str(CD_CLI).strip()
if not re.fullmatch(r'[+-]?[0-9]+', texto_cd_cli):
    raise TypeError('CD_CLI deve possuir representação inteira exata; valores fracionários são proibidos.')
CD_CLI = int(texto_cd_cli)
if not (-2147483648 <= CD_CLI <= 2147483647):
    raise ValueError('CD_CLI não cabe no tipo físico INT.')
print(f'[RADAR_MVP] CD_CLI validado: {CD_CLI}')


## Mapa funcional estático CATEGORIAS

A chave de casamento é exclusivamente CD_CATEGORIA + TIPO. As 70 entradas são locais ao contrato e não geram consulta externa.


In [ ]:
%%spark

COLUNAS_CATEGORIAS = [
    'TIPO', 'CD_GRUPO', 'TX_GRUPO', 'CD_CATEGORIA', 'TX_CATEGORIA',
    'CD_IR', 'TX_IR', 'CD_CLASS_RADAR', 'TX_CLASS_RADAR',
    'IN_AGRO', 'IN_PARTICIPA_CALCULO'
]

LINHAS_CATEGORIAS = [
    (None, 0, 'Sem categoria', 0, 'Sem categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S'),
    ('C', 1, 'Receitas', 1, 'Salário', 0, 'Não pertence', 1, 'Renda', 'N', 'S'),
    ('C', 1, 'Receitas', 2, 'Vale Alimentação', 0, 'Não pertence', 1, 'Renda', 'N', 'S'),
    ('C', 1, 'Receitas', 3, 'Restituição de IR', 0, 'Não pertence', 2, 'Estorno', 'N', 'S'),
    ('C', 1, 'Receitas', 4, 'Bonificação', 0, 'Não pertence', 1, 'Renda', 'N', 'S'),
    ('C', 1, 'Receitas', 5, 'Outros Rendimentos', 0, 'Não pertence', 1, 'Renda', 'N', 'S'),
    ('D', 2, 'Casa', 6, 'Água', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 2, 'Casa', 7, 'Eletricidade e Gás', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 2, 'Casa', 9, 'Compra de Imóvel', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S'),
    ('D', 2, 'Casa', 10, 'Aluguel e Condomínio', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 2, 'Casa', 11, 'Móveis e Utensílios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 2, 'Casa', 12, 'Serviços e Manutenção', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 2, 'Casa', 13, 'Empregados', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 2, 'Casa', 14, 'Animais e Pets', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 3, 'Educação', 15, 'Educação Superior', 1, 'Pagamentos efetuados', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 3, 'Educação', 16, 'Colégio', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S'),
    ('D', 3, 'Educação', 17, 'Idiomas', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 3, 'Educação', 18, 'Publicações e Papelaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 3, 'Educação', 20, 'Outros Gastos, Educação', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 4, 'Lazer', 21, 'Viagens e Lazer', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 4, 'Lazer', 22, 'Esportes e Academia', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 4, 'Lazer', 25, 'Cultura e Entretenimento', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 5, 'Saúde', 27, 'Plano de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S'),
    ('D', 5, 'Saúde', 28, 'Serviços de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S'),
    ('D', 5, 'Saúde', 29, 'Dentista', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S'),
    ('D', 5, 'Saúde', 30, 'Farmácias e Drogarias', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 6, 'Alimentação', 32, 'Feira e Supermercado', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 6, 'Alimentação', 35, 'Bar, Rest. e Padaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 7, 'Transporte', 36, 'Compra de Veículo', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S'),
    ('D', 7, 'Transporte', 37, 'Combustível', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 7, 'Transporte', 38, 'Estacionamento e Pedágio', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 7, 'Transporte', 39, 'Seguro de Veículo', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 7, 'Transporte', 40, 'Serviços e Manutenção', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 7, 'Transporte', 41, 'Transporte Urbano e Apps', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 42, 'Vestuário e Acessórios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 43, 'Cuidado Pessoal e Beleza', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 44, 'Compras Diversas', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 45, 'Pensão Alimentícia', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 46, 'Seguros e Previdência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 47, 'Doação', 4, 'Doações efetuadas', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 48, 'Gasto com Familiares', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 49, 'Presentes', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 9, 'Comunicação', 51, 'Telefonia e Internet', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 9, 'Comunicação', 53, 'Assinatura TV e Streaming', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 10, 'Tarifas e impostos', 54, 'IPTU', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 10, 'Tarifas e impostos', 55, 'IPVA e Gastos Detran', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 10, 'Tarifas e impostos', 56, 'Imposto de Renda', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 10, 'Tarifas e impostos', 57, 'ISS(Imposto sobre Serviços)', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S'),
    ('D', 10, 'Tarifas e impostos', 58, 'GPS(Guia de Previdência Social)', 0, 'Não pertence', 8, 'Futuro', 'N', 'S'),
    ('D', 10, 'Tarifas e impostos', 59, 'Serviços Financeiros', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 60, 'Serviços Diversos', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 4, 'Lazer', 61, 'Jogos e Loterias', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    (None, 0, 'Sem categoria', 83, 'Sem Categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S'),
    ('D', 12, 'Fatura', 111, 'Cartão de Crédito', 0, 'Não pertence', 9, 'Obrigações', 'N', 'N'),
    ('D', 11, 'Outros', 279, 'Gastos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S'),
    ('C', 14, 'Agro', 300, 'Receitas Agro', 0, 'Não pertence', 1, 'Renda', 'S', 'N'),
    ('D', 14, 'Agro', 310, 'Criações', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N'),
    ('D', 14, 'Agro', 330, 'Cultivos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N'),
    ('D', 14, 'Agro', 350, 'Insumos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N'),
    ('D', 14, 'Agro', 370, 'Apoio Produtivo', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N'),
    ('D', 10, 'Tarifas e impostos', 3787, 'IOF', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S'),
    ('D', 10, 'Tarifas e impostos', 3788, 'Encargos e Tarifas', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S'),
    ('D', 2, 'Casa', 3790, 'Seguro Residencial', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S'),
    ('D', 8, 'Despesas Pessoais', 4417, 'Empréstimos e Prestações', 3, 'Dívidas e ônus reais', 9, 'Obrigações', 'N', 'S'),
    ('D', 11, 'Outros', 39434, 'Cheque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S'),
    ('D', 11, 'Outros', 39435, 'Saque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S'),
    ('D', 11, 'Outros', 39436, 'Transferência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S'),
    ('D', 11, 'Outros', 39437, 'Boletos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S'),
    ('D', 13, 'Investimentos', 448977, 'Aplicação', 0, 'Não pertence', 8, 'Futuro', 'N', 'S'),
    ('C', 13, 'Investimentos', 448978, 'Resgate de Investimentos', 0, 'Não pertence', 3, 'Resgate', 'N', 'S'),
]

schema_categorias = StructType([
    StructField('TIPO', StringType(), True),
    StructField('CD_GRUPO', IntegerType(), False),
    StructField('TX_GRUPO', StringType(), False),
    StructField('CD_CATEGORIA', IntegerType(), False),
    StructField('TX_CATEGORIA', StringType(), False),
    StructField('CD_IR', IntegerType(), False),
    StructField('TX_IR', StringType(), False),
    StructField('CD_CLASS_RADAR', IntegerType(), False),
    StructField('TX_CLASS_RADAR', StringType(), False),
    StructField('IN_AGRO', StringType(), False),
    StructField('IN_PARTICIPA_CALCULO', StringType(), False),
])
df_categorias = spark.createDataFrame(LINHAS_CATEGORIAS, schema_categorias)


In [ ]:
%%spark

inicio = time.perf_counter()
if len(LINHAS_CATEGORIAS) != 70:
    raise AssertionError(f'CATEGORIAS deve possuir 70 entradas; encontrado={len(LINHAS_CATEGORIAS)}.')
chaves = [(r[3], r[0]) for r in LINHAS_CATEGORIAS]
if len(chaves) != len(set(chaves)):
    raise AssertionError('A combinação (CD_CATEGORIA, TIPO) não é única.')
for tipo, _, _, _, _, _, _, classe, _, agro, participa in LINHAS_CATEGORIAS:
    if tipo in ('C', 'D'):
        if tipo == 'C' and classe not in (0, 1, 2, 3, 4):
            raise AssertionError('Classe Radar incompatível com natureza C.')
        if tipo == 'D' and classe not in (5, 6, 7, 8, 9):
            raise AssertionError('Classe Radar incompatível com natureza D.')
    if agro not in ('S', 'N') or participa not in ('S', 'N'):
        raise AssertionError('Indicador inválido em CATEGORIAS.')
medir_transformacao('VALIDACAO_CATEGORIAS', inicio, 70, len(COLUNAS_CATEGORIAS))
print('[RADAR_MVP] CATEGORIAS validado: 70 entradas e chave composta única.')


### V23 — Auditoria pré-Q1

**OBJETIVO:** obter os registros de formação do cliente.  
**FONTE:** `DB2GFP.TRAN_RLZD_INST_PCT`.  
**CHAVE:** `CD_CLI`.  
**PRÉ-CONDIÇÃO:** `CD_CLI` inteiro informado.  
**FILTROS:** cliente informado, `CD_TIP_PSS = 1` e janela original de `TS_INCL_TRAN`.  
**COLUNAS RETORNADAS:** cliente, timestamp, CPF e atributos de conta.  
**GRÃO ESPERADO:** registros físicos elegíveis do único cliente.


## Consulta externa 1 — dados do cliente


In [ ]:
%%spark

nome_query = 'Q1_DADOS_CLIENTE'
inicio_wall = datetime.datetime.now()
inicio_total = time.perf_counter()
try:
    sql_cliente = f"""
SELECT
    CD_CLI,
    TS_INCL_TRAN,
    NR_CPF_CNPJ_TITR,
    NR_AG_TITR,
    CD_CT_TITR,
    NR_MCA_PCT_OPB,
    CD_PRD
FROM {FONTE_TRAN}
WHERE CD_CLI = {CD_CLI}
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
"""
    inicio_preparacao = time.perf_counter()
    df_cliente_raw = conector_db2.sql(
        sql_cliente,
        fetchsize=FETCHSIZE,
        query_timeout=QUERY_TIMEOUT_SECONDS,
    )
    fim_preparacao = time.perf_counter()
    df_cliente_raw = df_cliente_raw.persist(StorageLevel.MEMORY_AND_DISK)
    inicio_materializacao = time.perf_counter()
    qt_cliente_raw = df_cliente_raw.count()
    fim_materializacao = time.perf_counter()
    fim_wall = datetime.datetime.now()
    registrar_medicao('QUERY', nome_query, FONTE_TRAN, inicio_wall.isoformat(), fim_wall.isoformat(),
        fim_materializacao - inicio_total, qt_cliente_raw, len(df_cliente_raw.columns), 'OK',
        preparacao=fim_preparacao - inicio_preparacao,
        materializacao=fim_materializacao - inicio_materializacao)
except Exception as exc:
    fim_wall = datetime.datetime.now()
    registrar_medicao('QUERY', nome_query, FONTE_TRAN, inicio_wall.isoformat(), fim_wall.isoformat(),
        time.perf_counter() - inicio_total, None, None, 'ERRO', str(exc))
    raise


In [ ]:
%%spark

print(next(m for m in METRICAS if m['ETAPA_QUERY'] == 'Q1_DADOS_CLIENTE'))


### V23 — Auditoria Q1: registros de formação

Evidência produzida exclusivamente a partir de `df_cliente_raw` já materializado.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    df_aud_q1_resumo = df_cliente_raw.agg(
        F.min('TS_INCL_TRAN').alias('MENOR_TS_INCL_TRAN'),
        F.max('TS_INCL_TRAN').alias('MAIOR_TS_INCL_TRAN'),
        F.max('TS_INCL_TRAN').alias('TS_INCL_TRAN_REF')
    )
    auditar_painel('Q1 — FORMAÇÃO DO CLIENTE', [
        ('QT_LINHAS', qt_cliente_raw), ('QT_COLUNAS', len(df_cliente_raw.columns))
    ])
    df_aud_q1_resumo.show(truncate=False)
    auditar_schema('df_cliente_raw', df_cliente_raw)
    auditar_valores_controlados('df_cliente_raw', df_cliente_raw, [
        'CD_CLI', 'NR_CPF_CNPJ_TITR', 'NR_MCA_PCT_OPB', 'CD_PRD', 'NR_AG_TITR', 'CD_CT_TITR'
    ])
    auditar_linhas('Q1 — registros recebidos', df_cliente_raw, qt_cliente_raw)
    registrar_auditoria('AUD_Q1_DADOS_CLIENTE', inicio_auditoria, qt_cliente_raw, len(df_cliente_raw.columns))


## Formação do cliente, CPF e conta elegível


In [ ]:
%%spark

inicio = time.perf_counter()
if qt_cliente_raw <= 0:
    raise AssertionError('O CD_CLI não pertence à janela de formação do público.')
if df_cliente_raw.filter(F.col('CD_CLI') != F.lit(CD_CLI)).limit(1).count() != 0:
    raise AssertionError('A consulta retornou CD_CLI diferente do solicitado.')
linha_formacao = df_cliente_raw.agg(F.max('TS_INCL_TRAN').alias('TS_INCL_TRAN_REF')).first()
if linha_formacao['TS_INCL_TRAN_REF'] is None:
    raise AssertionError('TS_INCL_TRAN_REF obrigatório não foi produzido.')
resultado = {
    'CD_CLI': CD_CLI,
    'DT_EXEA': DATA_EXECUCAO,
    'DT_MES_EXEA': DT_MES_EXEA,
    'TS_INCL_TRAN_REF': linha_formacao['TS_INCL_TRAN_REF'],
    'FL_TEM_MOV_AGRO': None,
}
medir_transformacao('FORMACAO_CLIENTE', inicio, 1, 4)


In [ ]:
%%spark

inicio = time.perf_counter()
cpf = df_cliente_raw.agg(
    F.countDistinct('NR_CPF_CNPJ_TITR').alias('QT_CPF'),
    F.max('NR_CPF_CNPJ_TITR').alias('CPF_UNICO')
).first()
resultado['FL_CPF_UNICO'] = 'S' if int(cpf['QT_CPF']) == 1 else 'N'
resultado['CD_CPF'] = Decimal(str(cpf['CPF_UNICO'])) if resultado['FL_CPF_UNICO'] == 'S' else None
medir_transformacao('CPF', inicio, 1, 2)


In [ ]:
%%spark

inicio = time.perf_counter()
df_contas_distintas = (
    df_cliente_raw
    .filter(
        (F.col('NR_MCA_PCT_OPB') == F.lit(999999999)) &
        (F.col('CD_PRD') == F.lit(6)) &
        F.col('NR_AG_TITR').isNotNull() &
        F.col('CD_CT_TITR').isNotNull() &
        (F.trim(F.col('CD_CT_TITR').cast('string')) != F.lit(''))
    )
    .select(
        F.col('NR_AG_TITR').alias('NR_AG_TITR'),
        F.col('CD_CT_TITR').alias('CD_CT_TITR')
    )
    .groupBy('NR_AG_TITR', 'CD_CT_TITR')
    .count()
)
contas = df_contas_distintas.collect()
resultado['FL_CONTA_ELEGIVEL_UNICA'] = 'S' if len(contas) == 1 else 'N'
conta_raw = contas[0] if len(contas) == 1 else None
medir_transformacao('CONTA_ELEGIVEL', inicio, 1, 1)


In [ ]:
%%spark

inicio = time.perf_counter()
conta_normalizada = None
if conta_raw is not None:
    texto_agencia = str(conta_raw['NR_AG_TITR']).strip()
    texto_conta = str(conta_raw['CD_CT_TITR']).strip()
    if re.fullmatch(r'[0-9]+', texto_agencia) and re.fullmatch(r'[0-9]+', texto_conta):
        agencia_num = int(texto_agencia)
        conta_significativa = texto_conta.lstrip('0') or '0'
        conta_num = int(conta_significativa)
        if -2147483648 <= agencia_num <= 2147483647 and len(conta_significativa) <= 11:
            conta_normalizada = (agencia_num, conta_num)
medir_transformacao('NORMALIZACAO_CONTA', inicio, 1 if conta_normalizada else 0, 2)
print(f'[RADAR_MVP] conta normalizada={conta_normalizada}')


### V23 — Auditoria de CPF e conta elegível

Mostra a decisão funcional e a normalização resultante, sem reaplicar nem alterar a regra.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    auditar_painel('CPF E CONTA ELEGÍVEL', [
        ('FL_CPF_UNICO', resultado['FL_CPF_UNICO']),
        ('CD_CPF', resultado['CD_CPF']),
        ('FL_CONTA_ELEGIVEL_UNICA', resultado['FL_CONTA_ELEGIVEL_UNICA']),
        ('CONTA_NORMALIZADA', conta_normalizada)
    ])
    df_aud_contas = (
        df_contas_distintas
        .select('NR_AG_TITR', 'CD_CT_TITR', F.col('count').alias('QT_POR_COMBINACAO'))
        .withColumn('CD_UOR_CC_NORMALIZADO', F.lit(conta_normalizada[0]).cast('int') if conta_normalizada else F.lit(None).cast('int'))
        .withColumn('NR_CC_NORMALIZADO', F.lit(conta_normalizada[1]).cast('decimal(11,0)') if conta_normalizada else F.lit(None).cast('decimal(11,0)'))
    )
    print('ORIGINAL: NR_AG_TITR | CD_CT_TITR | QT_POR_COMBINACAO')
    print('NORMALIZADO: CD_UOR_CC_NORMALIZADO | NR_CC_NORMALIZADO')
    auditar_linhas('Contas candidatas e normalização', df_aud_contas, len(contas))
    registrar_auditoria('AUD_CPF_CONTA', inicio_auditoria, len(contas), len(df_aud_contas.columns))


### V23 — Auditoria pré-Q2

**OBJETIVO:** consultar o ciclo da conta única normalizada.  
**FONTE:** `DB2GFP.CT_GRDR_FNCO`.  
**CHAVE:** `CD_UOR_CC + NR_CC`.  
**PRÉ-CONDIÇÃO:** conta elegível única e normalizada.  
**FILTROS:** exatamente a agência e a conta normalizadas pela etapa funcional.  
**COLUNAS RETORNADAS:** chave, `DD_INC_MM_CLC_BLC` e `TS_ULT_EXEA_PSQ`.  
**GRÃO ESPERADO:** possíveis registros do ciclo da única conta; a seleção segue o contrato oficial.


## Consulta externa 2 — ciclo


In [ ]:
%%spark

nome_query = 'Q2_CICLO'
inicio_wall = datetime.datetime.now()
inicio_total = time.perf_counter()
if resultado['FL_CONTA_ELEGIVEL_UNICA'] != 'S' or conta_normalizada is None:
    schema_ciclo_raw = StructType([
        StructField('CD_UOR_CC', IntegerType(), True),
        StructField('NR_CC', DecimalType(11, 0), True),
        StructField('DD_INC_MM_CLC_BLC', ShortType(), True),
        StructField('TS_ULT_EXEA_PSQ', TimestampType(), True),
    ])
    df_ciclo_raw = spark.createDataFrame([], schema_ciclo_raw)
    qt_ciclo_raw = 0
    fim_wall = datetime.datetime.now()
    registrar_medicao('QUERY', nome_query, FONTE_CICLO, inicio_wall.isoformat(), fim_wall.isoformat(),
        time.perf_counter() - inicio_total, 0, 4, 'SKIPPED', 'Conta única normalizada indisponível.')
else:
    try:
        cd_uor_cc, nr_cc = conta_normalizada
        sql_ciclo = f"""
SELECT
    CD_UOR_CC,
    NR_CC,
    DD_INC_MM_CLC_BLC,
    TS_ULT_EXEA_PSQ
FROM {FONTE_CICLO}
WHERE CD_UOR_CC = {cd_uor_cc}
  AND NR_CC = {nr_cc}
"""
        inicio_preparacao = time.perf_counter()
        df_ciclo_raw = conector_db2.sql(sql_ciclo, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS)
        fim_preparacao = time.perf_counter()
        df_ciclo_raw = df_ciclo_raw.persist(StorageLevel.MEMORY_AND_DISK)
        inicio_materializacao = time.perf_counter()
        qt_ciclo_raw = df_ciclo_raw.count()
        fim_materializacao = time.perf_counter()
        fim_wall = datetime.datetime.now()
        registrar_medicao('QUERY', nome_query, FONTE_CICLO, inicio_wall.isoformat(), fim_wall.isoformat(),
            fim_materializacao - inicio_total, qt_ciclo_raw, len(df_ciclo_raw.columns), 'OK',
            preparacao=fim_preparacao - inicio_preparacao,
            materializacao=fim_materializacao - inicio_materializacao)
    except Exception as exc:
        fim_wall = datetime.datetime.now()
        registrar_medicao('QUERY', nome_query, FONTE_CICLO, inicio_wall.isoformat(), fim_wall.isoformat(),
            time.perf_counter() - inicio_total, None, None, 'ERRO', str(exc))
        raise


In [ ]:
%%spark

print(next(m for m in METRICAS if m['ETAPA_QUERY'] == 'Q2_CICLO'))


In [ ]:
%%spark

inicio = time.perf_counter()
linha_ciclo = None
if qt_ciclo_raw > 0:
    janela_ciclo = Window.partitionBy('CD_UOR_CC', 'NR_CC').orderBy(F.col('TS_ULT_EXEA_PSQ').desc())
    linha_ciclo = (
        df_ciclo_raw.withColumn('_RN', F.row_number().over(janela_ciclo))
        .filter(F.col('_RN') == 1)
        .select('TS_ULT_EXEA_PSQ', 'DD_INC_MM_CLC_BLC')
        .first()
    )
resultado['TS_DD_INC_MM_CLC_BLC_REF'] = linha_ciclo['TS_ULT_EXEA_PSQ'] if linha_ciclo else None
resultado['DD_INC_MM_CLC_BLC'] = int(linha_ciclo['DD_INC_MM_CLC_BLC']) if linha_ciclo and linha_ciclo['DD_INC_MM_CLC_BLC'] is not None else None
if resultado['FL_CONTA_ELEGIVEL_UNICA'] == 'N':
    resultado['DD_INC_MM_CLC_BLC_FALLBACK'] = None
elif linha_ciclo is None:
    resultado['DD_INC_MM_CLC_BLC_FALLBACK'] = 1
else:
    resultado['DD_INC_MM_CLC_BLC_FALLBACK'] = resultado['DD_INC_MM_CLC_BLC']
medir_transformacao('CICLO', inicio, 1, 3)


In [ ]:
%%spark

inicio = time.perf_counter()
dia_ciclo = resultado['DD_INC_MM_CLC_BLC_FALLBACK']
if dia_ciclo is None:
    resultado['DT_REF_INI'] = None
    resultado['DT_REF_FIM'] = None
else:
    if not (1 <= int(dia_ciclo) <= 31):
        raise AssertionError(f'Dia de ciclo fora do domínio 1..31: {dia_ciclo}.')
    ts_ref = resultado['TS_INCL_TRAN_REF']
    dia_mes_ref = min(int(dia_ciclo), calendar.monthrange(ts_ref.year, ts_ref.month)[1])
    candidato = datetime.datetime(ts_ref.year, ts_ref.month, dia_mes_ref)
    if ts_ref >= candidato:
        inicio_aberto = candidato.date()
    else:
        total = ts_ref.year * 12 + ts_ref.month - 2
        ano_ant, mes_ant_zero = divmod(total, 12)
        mes_ant = mes_ant_zero + 1
        inicio_aberto = datetime.date(ano_ant, mes_ant, min(int(dia_ciclo), calendar.monthrange(ano_ant, mes_ant)[1]))
    resultado['DT_REF_FIM'] = inicio_aberto - timedelta(days=1)
    total_fechado = inicio_aberto.year * 12 + inicio_aberto.month - 2
    ano_fechado, mes_fechado_zero = divmod(total_fechado, 12)
    mes_fechado = mes_fechado_zero + 1
    resultado['DT_REF_INI'] = datetime.date(
        ano_fechado, mes_fechado,
        min(int(dia_ciclo), calendar.monthrange(ano_fechado, mes_fechado)[1])
    )
medir_transformacao('JANELA_FINANCEIRA', inicio, 1, 2)
print(f"[RADAR_MVP] janela financeira={resultado['DT_REF_INI']}..{resultado['DT_REF_FIM']}")


### V23 — Auditoria Q2: ciclo e janela financeira

A ordenação abaixo é apenas de visualização; a seleção funcional permanece inalterada.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    auditar_painel('Q2 — CICLO E JANELA FINANCEIRA', [
        ('QT_LINHAS_CICLO', qt_ciclo_raw),
        ('DATA_EXECUCAO', DATA_EXECUCAO),
        ('DATA_INICIAL_PUBLICO', DATA_INICIAL_PUBLICO),
        ('DATA_FINAL_EXCLUSIVA_PUBLICO', DATA_FINAL_EXCLUSIVA_PUBLICO),
        ('TS_INCL_TRAN_REF', resultado['TS_INCL_TRAN_REF']),
        ('TS_DD_INC_MM_CLC_BLC_REF', resultado['TS_DD_INC_MM_CLC_BLC_REF']),
        ('DD_INC_MM_CLC_BLC', resultado['DD_INC_MM_CLC_BLC']),
        ('DD_INC_MM_CLC_BLC_FALLBACK', resultado['DD_INC_MM_CLC_BLC_FALLBACK']),
        ('DT_REF_INI', resultado['DT_REF_INI']),
        ('DT_REF_FIM', resultado['DT_REF_FIM'])
    ])
    auditar_schema('df_ciclo_raw', df_ciclo_raw)
    df_aud_ciclo_ordenado = df_ciclo_raw.orderBy(F.col('TS_ULT_EXEA_PSQ').desc())
    auditar_linhas('Q2 — registros de ciclo ordenados para auditoria', df_aud_ciclo_ordenado, qt_ciclo_raw)
    registrar_auditoria('AUD_Q2_CICLO_JANELA', inicio_auditoria, qt_ciclo_raw, len(df_ciclo_raw.columns))


### V23 — Auditoria pré-Q3

**OBJETIVO:** obter os registros de renda do CPF único.  
**FONTE:** `DB2DFE.REN_AVLD_PF`, lida pelo Hive.  
**CHAVE:** `NR_CPF_BASE_SRF`.  
**PRÉ-CONDIÇÃO:** CPF único disponível.  
**FILTROS:** somente o CPF selecionado; nenhum adicional.  
**COLUNAS RETORNADAS:** `NR_CPF_BASE_SRF AS NR_CPF`, `DT_INCL_REN_AVLD`, `VL_REN`.  
**GRÃO ESPERADO:** histórico de renda do CPF.


## Consulta externa 3 — renda presumida

NR_CPF_BASE_SRF é o atributo físico do CPF utilizado na consulta de renda; a consulta o retorna como NR_CPF.


In [ ]:
%%spark

nome_query = 'Q3_RENDA'
inicio_wall = datetime.datetime.now()
inicio_total = time.perf_counter()
if resultado['FL_CPF_UNICO'] != 'S' or resultado['CD_CPF'] is None:
    schema_renda_raw = SCHEMA_Q3_SKIPPED
    df_renda_raw = spark.createDataFrame([], schema_renda_raw)
    qt_renda_raw = 0
    fim_wall = datetime.datetime.now()
    registrar_medicao('QUERY', nome_query, FONTE_RENDA, inicio_wall.isoformat(), fim_wall.isoformat(),
        time.perf_counter() - inicio_total, 0, 3, 'SKIPPED', 'CPF único indisponível.')
else:
    try:
        cpf_sql = int(resultado['CD_CPF'])
        sql_renda = f"""
SELECT
    NR_CPF_BASE_SRF AS NR_CPF,
    DT_INCL_REN_AVLD,
    VL_REN
FROM {FONTE_RENDA}
WHERE NR_CPF_BASE_SRF = {cpf_sql}
"""
        inicio_preparacao = time.perf_counter()
        df_renda_raw = spark.sql(sql_renda)
        fim_preparacao = time.perf_counter()
        df_renda_raw = df_renda_raw.persist(StorageLevel.MEMORY_AND_DISK)
        inicio_materializacao = time.perf_counter()
        qt_renda_raw = df_renda_raw.count()
        fim_materializacao = time.perf_counter()
        fim_wall = datetime.datetime.now()
        registrar_medicao('QUERY', nome_query, FONTE_RENDA, inicio_wall.isoformat(), fim_wall.isoformat(),
            fim_materializacao - inicio_total, qt_renda_raw, len(df_renda_raw.columns), 'OK',
            preparacao=fim_preparacao - inicio_preparacao,
            materializacao=fim_materializacao - inicio_materializacao)
    except Exception as exc:
        fim_wall = datetime.datetime.now()
        registrar_medicao('QUERY', nome_query, FONTE_RENDA, inicio_wall.isoformat(), fim_wall.isoformat(),
            time.perf_counter() - inicio_total, None, None, 'ERRO', str(exc))
        raise

if resultado['FL_CPF_UNICO'] != 'S' or resultado['CD_CPF'] is None:
    STATUS_SCHEMA_Q3 = 'SKIPPED'
else:
    STATUS_SCHEMA_Q3 = comparar_schema_referencia('Q3_RENDA', df_renda_raw, SCHEMA_Q3_SKIPPED)
    if STATUS_SCHEMA_Q3 != 'OK':
        raise AssertionError(f'Gate de schema Q3 falhou: STATUS_SCHEMA_Q3={STATUS_SCHEMA_Q3}.')
print(f'[V23][GATE] STATUS_SCHEMA_Q3={STATUS_SCHEMA_Q3}')


In [ ]:
%%spark

print(next(m for m in METRICAS if m['ETAPA_QUERY'] == 'Q3_RENDA'))


In [ ]:
%%spark

inicio = time.perf_counter()
linha_renda = None
if qt_renda_raw > 0:
    janela_renda = Window.partitionBy('NR_CPF').orderBy(F.col('DT_INCL_REN_AVLD').desc())
    linha_renda = (
        df_renda_raw.withColumn('_RN', F.row_number().over(janela_renda))
        .filter(F.col('_RN') == 1)
        .select(F.to_date('DT_INCL_REN_AVLD').alias('DT_INCL_REN_AVLD'), F.col('VL_REN'))
        .first()
    )
resultado['DT_REN_PRES_REF'] = linha_renda['DT_INCL_REN_AVLD'] if linha_renda else None
resultado['VL_REN_PRES'] = linha_renda['VL_REN'] if linha_renda else None
medir_transformacao('RENDA', inicio, 1, 2)


### V23 — Auditoria Q3: renda presumida

Evidência do CPF efetivamente utilizado, dos registros retornados e da linha selecionada.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    auditar_painel('Q3 — RENDA PRESUMIDA', [
        ('NR_CPF_BASE_SRF_UTILIZADO', resultado['CD_CPF']),
        ('QT_LINHAS_RENDA', qt_renda_raw),
        ('DT_REN_PRES_REF', resultado['DT_REN_PRES_REF']),
        ('VL_REN_PRES', resultado['VL_REN_PRES'])
    ])
    auditar_schema('df_renda_raw', df_renda_raw)
    print(f'[AUDITORIA] STATUS_SCHEMA_Q3={STATUS_SCHEMA_Q3}')
    df_aud_renda_ordenada = df_renda_raw.orderBy(F.col('DT_INCL_REN_AVLD').desc())
    auditar_linhas('Q3 — histórico de renda ordenado para auditoria', df_aud_renda_ordenada, qt_renda_raw)
    registrar_auditoria('AUD_Q3_RENDA', inicio_auditoria, qt_renda_raw, len(df_renda_raw.columns))


### V23 — Auditoria pré-Q4

**OBJETIVO:** recuperar o perfil financeiro elegível na data de execução.  
**FONTE:** `DB2D1D.DVS_GRDR_FNCO_PF`.  
**CHAVE:** `CD_CLI`.  
**PRÉ-CONDIÇÃO:** cliente válido.  
**FILTROS:** `CD_CLI` informado e `DT_REF <= DATA_EXECUCAO`.  
**COLUNAS RETORNADAS:** data de referência e códigos/nomes de macro e microperfil.  
**GRÃO ESPERADO:** registros elegíveis do cliente; a maior `DT_REF` é selecionada no Spark, sem desempate por perfil ou nome.


## Consulta externa 4 — perfil financeiro


In [ ]:
%%spark

nome_query = 'Q4_PERFIL'
inicio_wall = datetime.datetime.now()
inicio_total = time.perf_counter()
try:
    sql_perfil = f"""
SELECT
    CD_CLI,
    DT_REF,
    CD_MAC_PRFL_CLI,
    NM_MAC_PRFL_CLI,
    CD_MIC_PRFL_CLI,
    NM_MIC_PRFL_CLI
FROM {FONTE_PERFIL}
WHERE CD_CLI = {CD_CLI}
  AND DT_REF <= DATE('{DATA_EXECUCAO.isoformat()}')
"""
    inicio_preparacao = time.perf_counter()
    df_perfil_raw = conector_db2.sql(sql_perfil, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS)
    fim_preparacao = time.perf_counter()
    df_perfil_raw = df_perfil_raw.persist(StorageLevel.MEMORY_AND_DISK)
    inicio_materializacao = time.perf_counter()
    qt_perfil_raw = df_perfil_raw.count()
    fim_materializacao = time.perf_counter()
    fim_wall = datetime.datetime.now()
    registrar_medicao('QUERY', nome_query, FONTE_PERFIL, inicio_wall.isoformat(), fim_wall.isoformat(),
        fim_materializacao - inicio_total, qt_perfil_raw, len(df_perfil_raw.columns), 'OK',
        preparacao=fim_preparacao - inicio_preparacao,
        materializacao=fim_materializacao - inicio_materializacao)
except Exception as exc:
    fim_wall = datetime.datetime.now()
    registrar_medicao('QUERY', nome_query, FONTE_PERFIL, inicio_wall.isoformat(), fim_wall.isoformat(),
        time.perf_counter() - inicio_total, None, None, 'ERRO', str(exc))
    raise


In [ ]:
%%spark

print(next(m for m in METRICAS if m['ETAPA_QUERY'] == 'Q4_PERFIL'))


In [ ]:
%%spark

inicio = time.perf_counter()
linha_perfil = None
qt_perfil_selecionado = 0
if qt_perfil_raw > 0:
    dt_ref_selecionada = df_perfil_raw.agg(F.max('DT_REF').alias('DT_REF')).first()['DT_REF']
    linhas_perfil_selecionadas = (
        df_perfil_raw
        .filter(F.col('DT_REF').eqNullSafe(F.lit(dt_ref_selecionada)))
        .select('DT_REF', 'CD_MAC_PRFL_CLI', 'NM_MAC_PRFL_CLI', 'CD_MIC_PRFL_CLI', 'NM_MIC_PRFL_CLI')
        .take(2)
    )
    qt_perfil_selecionado = len(linhas_perfil_selecionadas)
    if qt_perfil_selecionado > 1:
        raise AssertionError(
            f'Q4 retornou mais de uma linha na maior DT_REF elegível: {dt_ref_selecionada}.'
        )
    linha_perfil = linhas_perfil_selecionadas[0] if linhas_perfil_selecionadas else None
if linha_perfil and linha_perfil['DT_REF'] > DATA_EXECUCAO:
    raise AssertionError(
        f'DT_REF_PRFL posterior à DATA_EXECUCAO: {linha_perfil["DT_REF"]} > {DATA_EXECUCAO}.'
    )
resultado['DT_REF_PRFL'] = linha_perfil['DT_REF'] if linha_perfil else None
resultado['CD_MAC_PRFL_CLI'] = int(linha_perfil['CD_MAC_PRFL_CLI']) if linha_perfil and linha_perfil['CD_MAC_PRFL_CLI'] is not None else None
resultado['NM_MAC_PRFL_CLI'] = linha_perfil['NM_MAC_PRFL_CLI'] if linha_perfil else None
resultado['CD_MIC_PRFL_CLI'] = int(linha_perfil['CD_MIC_PRFL_CLI']) if linha_perfil and linha_perfil['CD_MIC_PRFL_CLI'] is not None else None
resultado['NM_MIC_PRFL_CLI'] = linha_perfil['NM_MIC_PRFL_CLI'] if linha_perfil else None
medir_transformacao('PERFIL', inicio, qt_perfil_selecionado, 5)


### V23 — Auditoria Q4: perfil financeiro

Mostra os registros elegíveis e a linha da maior `DT_REF`, usando apenas `df_perfil_raw` já materializado.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    auditar_painel('Q4 — PERFIL ELEGÍVEL SELECIONADO', [
        ('QT_LINHAS_Q4_RAW', qt_perfil_raw),
        ('QT_LINHAS_PERFIL', qt_perfil_selecionado),
        ('DT_REF_CORTE', DATA_EXECUCAO),
        ('DT_REF_PRFL', resultado['DT_REF_PRFL']),
        ('CD_MAC_PRFL_CLI', resultado['CD_MAC_PRFL_CLI']),
        ('NM_MAC_PRFL_CLI', resultado['NM_MAC_PRFL_CLI']),
        ('CD_MIC_PRFL_CLI', resultado['CD_MIC_PRFL_CLI']),
        ('NM_MIC_PRFL_CLI', resultado['NM_MIC_PRFL_CLI'])
    ])
    auditar_schema('df_perfil_raw', df_perfil_raw)
    auditar_valores_controlados('df_perfil_raw', df_perfil_raw, [
        'CD_MAC_PRFL_CLI', 'NM_MAC_PRFL_CLI', 'CD_MIC_PRFL_CLI', 'NM_MIC_PRFL_CLI'
    ])
    df_aud_perfil_ordenado = df_perfil_raw.orderBy(F.col('DT_REF').desc())
    auditar_linhas('Q4 — registros elegíveis ordenados por DT_REF', df_aud_perfil_ordenado, qt_perfil_raw)
    registrar_auditoria('AUD_Q4_PERFIL', inicio_auditoria, qt_perfil_selecionado, 5)


## Gate da consulta de movimentações


In [ ]:
%%spark

JANELA_FINANCEIRA_DISPONIVEL = resultado['DT_REF_INI'] is not None and resultado['DT_REF_FIM'] is not None
if (resultado['DT_REF_INI'] is None) != (resultado['DT_REF_FIM'] is None):
    raise AssertionError('DT_REF_INI e DT_REF_FIM devem existir conjuntamente.')
print(f'[RADAR_MVP] consulta de movimentações liberada={JANELA_FINANCEIRA_DISPONIVEL}')


### V23 — Evidência do gate da movimentação

Painel de autorização da consulta Q5; não executa nenhuma consulta adicional.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    status_q5 = 'AUTORIZADA' if JANELA_FINANCEIRA_DISPONIVEL else 'SKIPPED'
    print('=' * 60)
    print('QUERY DE MOVIMENTAÇÕES')
    print(f'CD_CLI     = {CD_CLI}')
    print(f"DT_REF_INI = {resultado['DT_REF_INI']}")
    print(f"DT_REF_FIM = {resultado['DT_REF_FIM']}")
    print(f'STATUS     = {status_q5}')
    print('=' * 60)
    registrar_auditoria('AUD_GATE_Q5', inicio_auditoria, 0, 0, status_q5)


### V23 — Auditoria pré-Q5

**OBJETIVO:** recuperar movimentações dentro da janela financeira já calculada.  
**FONTE:** `DB2GFP.TRAN_RLZD_INST_PCT`.  
**CHAVE:** `CD_CLI` e intervalo de `DT_TRAN`.  
**PRÉ-CONDIÇÃO:** `DT_REF_INI` e `DT_REF_FIM` existem conjuntamente.  
**FILTROS:** cliente, limites inclusivos da janela e `CD_EST_TRAN_INST = 0`.  
**ESCOPO DE CONTAS:** todas as contas e instituições do cliente; marca, produto, agência e conta não filtram a Q5.  
**ÂNCORA TEMPORAL:** Q1 forma o cliente por `TS_INCL_TRAN`; Q5 seleciona o evento econômico por `DT_TRAN`. Late arrival é permitido.  
**EXCLUSÕES:** transações fora do estado obrigatório são ignoradas antes da classificação e da sumarização.  
**COLUNAS RETORNADAS:** identificador técnico da transação, cliente, data, natureza, categoria, moeda e valor.  
**GRÃO ESPERADO:** transações físicas do cliente na janela.


## Consulta externa 5 — movimentações


In [ ]:
%%spark

nome_query = 'Q5_MOVIMENTACOES'
inicio_wall = datetime.datetime.now()
inicio_total = time.perf_counter()
qt_nao_brl = None
if not JANELA_FINANCEIRA_DISPONIVEL:
    schema_mov_raw = SCHEMA_Q5_SKIPPED
    df_mov_raw = spark.createDataFrame([], schema_mov_raw)
    qt_mov_raw = 0
    fim_wall = datetime.datetime.now()
    registrar_medicao('QUERY', nome_query, FONTE_TRAN, inicio_wall.isoformat(), fim_wall.isoformat(),
        time.perf_counter() - inicio_total, 0, 7, 'SKIPPED', 'DT_REF_INI/DT_REF_FIM indisponíveis.')
else:
    try:
        sql_mov = f"""
SELECT
    NR_TRAN_INST_PCT,
    CD_CLI,
    DT_TRAN,
    CD_NTZ_CTB_TRAN,
    CD_CTGR_TRAN_OGNL,
    CD_TIP_MOE_CRR,
    VL_TRAN
FROM {FONTE_TRAN}
WHERE CD_CLI = {CD_CLI}
  AND DT_TRAN >= DATE('{resultado['DT_REF_INI'].isoformat()}')
  AND DT_TRAN <= DATE('{resultado['DT_REF_FIM'].isoformat()}')
  AND CD_EST_TRAN_INST = 0
"""
        inicio_preparacao = time.perf_counter()
        df_mov_raw = conector_db2.sql(sql_mov, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS)
        fim_preparacao = time.perf_counter()
        df_mov_raw = df_mov_raw.persist(StorageLevel.MEMORY_AND_DISK)
        inicio_materializacao = time.perf_counter()
        qt_mov_raw = df_mov_raw.count()
        fim_materializacao = time.perf_counter()
        fim_wall = datetime.datetime.now()
        registrar_medicao('QUERY', nome_query, FONTE_TRAN, inicio_wall.isoformat(), fim_wall.isoformat(),
            fim_materializacao - inicio_total, qt_mov_raw, len(df_mov_raw.columns), 'OK',
            preparacao=fim_preparacao - inicio_preparacao,
            materializacao=fim_materializacao - inicio_materializacao)
    except Exception as exc:
        fim_wall = datetime.datetime.now()
        registrar_medicao('QUERY', nome_query, FONTE_TRAN, inicio_wall.isoformat(), fim_wall.isoformat(),
            time.perf_counter() - inicio_total, None, None, 'ERRO', str(exc))
        raise

if not JANELA_FINANCEIRA_DISPONIVEL:
    STATUS_SCHEMA_Q5 = 'SKIPPED'
else:
    STATUS_SCHEMA_Q5 = comparar_schema_referencia('Q5_MOVIMENTACOES', df_mov_raw, SCHEMA_Q5_SKIPPED)
    if STATUS_SCHEMA_Q5 != 'OK':
        raise AssertionError(f'Gate de schema Q5 falhou: STATUS_SCHEMA_Q5={STATUS_SCHEMA_Q5}.')
print(f'[V23][GATE] STATUS_SCHEMA_Q5={STATUS_SCHEMA_Q5}')


In [ ]:
%%spark

print(next(m for m in METRICAS if m['ETAPA_QUERY'] == 'Q5_MOVIMENTACOES'))


### V23 — Auditoria Q5: movimentações recebidas

Diagnóstico restrito ao universo já consultado para o cliente; não há leitura global.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    df_aud_mov_resumo = df_mov_raw.agg(
        F.min('DT_TRAN').alias('MENOR_DT_TRAN'),
        F.max('DT_TRAN').alias('MAIOR_DT_TRAN'),
        F.count(F.lit(1)).alias('QT_VL_TRAN'),
        F.sum(F.when(F.col('VL_TRAN').isNull(), 1).otherwise(0)).alias('QT_VL_TRAN_NULL'),
        F.min('VL_TRAN').alias('MENOR_VL_TRAN'),
        F.max('VL_TRAN').alias('MAIOR_VL_TRAN'),
        F.sum('VL_TRAN').alias('SOMA_VL_TRAN')
    )
    auditar_painel('Q5 — MOVIMENTAÇÕES', [
        ('QT_LINHAS', qt_mov_raw), ('QT_COLUNAS', len(df_mov_raw.columns)),
        ('DT_REF_INI', resultado['DT_REF_INI']), ('DT_REF_FIM', resultado['DT_REF_FIM'])
    ])
    df_aud_mov_resumo.show(truncate=False)
    if JANELA_FINANCEIRA_DISPONIVEL:
        df_aud_fora_janela = df_mov_raw.filter((F.col('DT_TRAN') < F.lit(resultado['DT_REF_INI'])) | (F.col('DT_TRAN') > F.lit(resultado['DT_REF_FIM'])))
        qt_fora_janela = df_aud_fora_janela.count()
        print(f'[AUDITORIA] MIN(DT_TRAN) >= DT_REF_INI e MAX(DT_TRAN) <= DT_REF_FIM: {qt_fora_janela == 0}')
    auditar_schema('df_mov_raw', df_mov_raw)
    print(f'[AUDITORIA] STATUS_SCHEMA_Q5={STATUS_SCHEMA_Q5}')
    auditar_valores_controlados('df_mov_raw', df_mov_raw, ['CD_NTZ_CTB_TRAN', 'CD_CTGR_TRAN_OGNL', 'CD_TIP_MOE_CRR'])
    df_aud_mov_ordenado = df_mov_raw.orderBy(F.col('DT_TRAN').asc_nulls_last())
    auditar_linhas('Q5 — movimentações ordenadas para auditoria', df_aud_mov_ordenado, qt_mov_raw)
    registrar_auditoria('AUD_Q5_MOVIMENTACOES', inicio_auditoria, qt_mov_raw, len(df_mov_raw.columns))


## Anulação quantitativa de crédito e débito

A anulação ocorre no Spark, sobre a Q5 já materializada, e antes da classificação. A chave exata é `CD_CLI + DT_TRAN + VL_TRAN + CD_TIP_MOE_CRR`. Para cada chave, removem-se `MIN(QT_C, QT_D)` registros de cada natureza. `NR_TRAN_INST_PCT ASC` é somente um critério técnico determinístico, sem significado negocial.


In [ ]:
%%spark

inicio = time.perf_counter()
CHAVE_ANULACAO = ['CD_CLI', 'DT_TRAN', 'VL_TRAN', 'CD_TIP_MOE_CRR']
COLUNAS_MOV_FUNCIONAIS = [
    'CD_CLI', 'DT_TRAN', 'CD_NTZ_CTB_TRAN',
    'CD_CTGR_TRAN_OGNL', 'CD_TIP_MOE_CRR', 'VL_TRAN'
]

if not JANELA_FINANCEIRA_DISPONIVEL:
    df_chaves_anulacao = None
    df_mov_marcado_anulacao = None
    df_mov_efetivo = df_mov_raw.select(*COLUNAS_MOV_FUNCIONAIS)
    qt_chaves_com_anulacao = 0
    qt_pares_anulados = 0
    qt_transacoes_removidas = 0
    qt_mov_efetivo = 0
    qt_chaves_com_pares_residuais = 0
    STATUS_GATE_ANULACAO = 'SKIPPED'
    medir_transformacao(
        'ANULACAO_QUANTITATIVA_C_D', inicio, 0, 6, 'SKIPPED',
        'DT_REF_INI/DT_REF_FIM indisponíveis.'
    )
else:
    elegivel_anulacao = (
        F.col('CD_NTZ_CTB_TRAN').isin('C', 'D') &
        F.col('DT_TRAN').isNotNull() &
        F.col('VL_TRAN').isNotNull() &
        F.col('CD_TIP_MOE_CRR').isNotNull()
    )
    # Primeiro detecta quantitativamente as chaves. Não persiste este intermediário
    # enquanto não houver prova de que existem pares.
    df_chaves_anulacao_candidatas = (
        df_mov_raw
        .filter(elegivel_anulacao)
        .groupBy(*CHAVE_ANULACAO)
        .agg(
            F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == F.lit('C'), 1).otherwise(0)).cast('long').alias('QT_C'),
            F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == F.lit('D'), 1).otherwise(0)).cast('long').alias('QT_D')
        )
        .withColumn('QT_PARES_ANULADOS', F.least(F.col('QT_C'), F.col('QT_D')).cast('long'))
        .filter(F.col('QT_PARES_ANULADOS') > F.lit(0))
    )
    resumo_anulacao = df_chaves_anulacao_candidatas.agg(
        F.count(F.lit(1)).cast('long').alias('QT_CHAVES_COM_ANULACAO'),
        F.sum('QT_PARES_ANULADOS').cast('long').alias('QT_PARES_ANULADOS')
    ).first()
    qt_chaves_com_anulacao = int(resumo_anulacao['QT_CHAVES_COM_ANULACAO'] or 0)
    qt_pares_anulados = int(resumo_anulacao['QT_PARES_ANULADOS'] or 0)

    if qt_pares_anulados == 0:
        # FAST PATH: nenhuma segunda persistência, join, Window, sort ou ROW_NUMBER.
        df_chaves_anulacao = None
        df_mov_marcado_anulacao = None
        df_mov_efetivo = df_mov_raw.select(*COLUNAS_MOV_FUNCIONAIS)
        qt_mov_efetivo = qt_mov_raw
        qt_transacoes_removidas = 0
    else:
        # Com pares, materializa as poucas chaves afetadas porque serão reutilizadas.
        df_chaves_anulacao = df_chaves_anulacao_candidatas.persist(StorageLevel.MEMORY_AND_DISK)
        janela_anulacao = Window.partitionBy(
            *(CHAVE_ANULACAO + ['CD_NTZ_CTB_TRAN'])
        ).orderBy(F.col('NR_TRAN_INST_PCT').asc())
        df_mov_marcado_anulacao = (
            df_mov_raw
            .join(
                df_chaves_anulacao.select(*CHAVE_ANULACAO, 'QT_PARES_ANULADOS'),
                CHAVE_ANULACAO,
                'left'
            )
            .withColumn('_RN_ANULACAO', F.row_number().over(janela_anulacao))
            .withColumn(
                '_FL_ANULADA',
                F.when(
                    F.col('QT_PARES_ANULADOS').isNotNull() &
                    F.col('CD_NTZ_CTB_TRAN').isin('C', 'D') &
                    (F.col('_RN_ANULACAO') <= F.col('QT_PARES_ANULADOS')),
                    F.lit(True)
                ).otherwise(F.lit(False))
            )
        )
        df_mov_efetivo = (
            df_mov_marcado_anulacao
            .filter(~F.col('_FL_ANULADA'))
            .select(*COLUNAS_MOV_FUNCIONAIS)
            .persist(StorageLevel.MEMORY_AND_DISK)
        )
        qt_mov_efetivo = df_mov_efetivo.count()
        qt_transacoes_removidas = qt_mov_raw - qt_mov_efetivo

    STATUS_GATE_ANULACAO = 'DIVERGENTE'
    if qt_transacoes_removidas != 2 * qt_pares_anulados:
        raise AssertionError(
            'Anulação C/D inconsistente: '
            f'removidas={qt_transacoes_removidas}; pares={qt_pares_anulados}.'
        )
    if qt_mov_efetivo != qt_mov_raw - qt_transacoes_removidas:
        raise AssertionError(
            'Cardinalidade inconsistente após anulação: '
            f'raw={qt_mov_raw}; removidas={qt_transacoes_removidas}; efetivas={qt_mov_efetivo}.'
        )
    if df_mov_efetivo.columns != COLUNAS_MOV_FUNCIONAIS:
        raise AssertionError(f'Schema funcional inesperado após anulação: {df_mov_efetivo.columns}')

    qt_chaves_com_pares_residuais = 0
    if qt_pares_anulados > 0:
        df_validacao_pos_anulacao = (
            df_mov_efetivo
            .filter(elegivel_anulacao)
            .join(df_chaves_anulacao.select(*CHAVE_ANULACAO), CHAVE_ANULACAO, 'inner')
            .groupBy(*CHAVE_ANULACAO)
            .agg(
                F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == F.lit('C'), 1).otherwise(0)).alias('QT_C'),
                F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == F.lit('D'), 1).otherwise(0)).alias('QT_D')
            )
            .withColumn('QT_PARES_RESIDUAIS', F.least(F.col('QT_C'), F.col('QT_D')))
            .filter(F.col('QT_PARES_RESIDUAIS') > F.lit(0))
        )
        qt_chaves_com_pares_residuais = df_validacao_pos_anulacao.count()
    if qt_chaves_com_pares_residuais != 0:
        raise AssertionError(
            'Gate pós-anulação falhou: '
            f'QT_CHAVES_COM_PARES_RESIDUAIS={qt_chaves_com_pares_residuais}.'
        )

    STATUS_GATE_ANULACAO = 'OK'
    medir_transformacao(
        'ANULACAO_QUANTITATIVA_C_D', inicio, qt_mov_efetivo, 6,
        detalhe=(
            f'chaves={qt_chaves_com_anulacao}; pares={qt_pares_anulados}; '
            f'transacoes_removidas={qt_transacoes_removidas}; '
            f'pares_residuais={qt_chaves_com_pares_residuais}; fast_path={qt_pares_anulados == 0}'
        )
    )

print(f'[V23][GATE] STATUS_GATE_ANULACAO={STATUS_GATE_ANULACAO}')


### V23 — Auditoria da anulação quantitativa C/D

Apresenta as quantidades removidas e as chaves funcionais que produziram pares, sem alimentar o pipeline funcional.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    auditar_painel('ANULAÇÃO QUANTITATIVA C/D', [
        ('QT_MOV_RAW', qt_mov_raw),
        ('QT_CHAVES_COM_ANULACAO', qt_chaves_com_anulacao),
        ('QT_PARES_ANULADOS', qt_pares_anulados),
        ('QT_TRANSACOES_REMOVIDAS', qt_transacoes_removidas),
        ('QT_MOV_EFETIVO', qt_mov_efetivo),
        ('QT_CHAVES_COM_PARES_RESIDUAIS', qt_chaves_com_pares_residuais),
        ('INVARIANTE_PARES', qt_transacoes_removidas == 2 * qt_pares_anulados),
        ('INVARIANTE_CARDINALIDADE', qt_mov_efetivo == qt_mov_raw - qt_transacoes_removidas)
    ])
    if df_chaves_anulacao is not None:
        df_aud_chaves_anulacao = df_chaves_anulacao.orderBy(
            F.col('CD_CLI'), F.col('DT_TRAN'), F.col('VL_TRAN'), F.col('CD_TIP_MOE_CRR')
        )
        auditar_linhas(
            'Chaves com pares C/D anulados',
            df_aud_chaves_anulacao,
            qt_chaves_com_anulacao
        )
    if df_mov_marcado_anulacao is not None and qt_transacoes_removidas > 0:
        df_aud_linhas_anuladas = (
            df_mov_marcado_anulacao
            .filter(F.col('_FL_ANULADA'))
            .select(
                'NR_TRAN_INST_PCT', 'DT_TRAN', 'CD_NTZ_CTB_TRAN',
                'CD_CTGR_TRAN_OGNL', 'CD_TIP_MOE_CRR', 'VL_TRAN'
            )
            .orderBy(
                F.col('DT_TRAN'), F.col('VL_TRAN'), F.col('CD_TIP_MOE_CRR'),
                F.col('CD_NTZ_CTB_TRAN'), F.col('NR_TRAN_INST_PCT')
            )
        )
        auditar_linhas(
            'Linhas físicas removidas pela anulação',
            df_aud_linhas_anuladas,
            qt_transacoes_removidas
        )
    else:
        print('[AUDITORIA] Nenhuma linha física removida pela anulação.')
    registrar_auditoria(
        'AUD_ANULACAO_QUANTITATIVA_C_D', inicio_auditoria,
        qt_chaves_com_anulacao, 8
    )


## Classificação exata, moeda e universo BRL


In [ ]:
%%spark

inicio = time.perf_counter()
if JANELA_FINANCEIRA_DISPONIVEL:
    df_natureza_valida = df_mov_efetivo.filter(
        F.col('CD_NTZ_CTB_TRAN').isin('C', 'D') &
        F.col('CD_CTGR_TRAN_OGNL').isNotNull()
    )
    df_classificado = (
        df_natureza_valida.alias('TRAN')
        .join(
            df_categorias.alias('CATEGORIAS'),
            (F.col('TRAN.CD_CTGR_TRAN_OGNL') == F.col('CATEGORIAS.CD_CATEGORIA')) &
            (F.col('TRAN.CD_NTZ_CTB_TRAN') == F.col('CATEGORIAS.TIPO')),
            'inner'
        )
        .select(
            'TRAN.CD_CLI', 'TRAN.DT_TRAN', 'TRAN.CD_NTZ_CTB_TRAN',
            'TRAN.CD_CTGR_TRAN_OGNL', 'TRAN.CD_TIP_MOE_CRR', 'TRAN.VL_TRAN',
            'CATEGORIAS.CD_CLASS_RADAR', 'CATEGORIAS.IN_AGRO',
            'CATEGORIAS.IN_PARTICIPA_CALCULO'
        )
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    qt_natureza_valida = df_natureza_valida.count()
    qt_classificado = df_classificado.count()
    if qt_classificado > qt_natureza_valida:
        raise AssertionError('O casamento com CATEGORIAS multiplicou transações.')
else:
    df_classificado = None
    qt_natureza_valida = 0
    qt_classificado = 0
medir_transformacao('CLASSIFICACAO_TRANSACIONAL', inicio, qt_classificado, 9)


In [ ]:
%%spark

inicio = time.perf_counter()
if not JANELA_FINANCEIRA_DISPONIVEL:
    resultado['FL_SOMENTE_BRL'] = None
    df_brl = None
elif qt_classificado == 0:
    resultado['FL_SOMENTE_BRL'] = 'N'
    df_brl = df_classificado.filter(F.lit(False))
else:
    moeda = df_classificado.agg(
        F.sum(F.when(F.col('CD_TIP_MOE_CRR').isNull() | (F.col('CD_TIP_MOE_CRR') == '') | (F.col('CD_TIP_MOE_CRR') != 'BRL'), 1).otherwise(0)).alias('QT_NAO_BRL')
    ).first()
    qt_nao_brl = int(moeda['QT_NAO_BRL'] or 0)
    resultado['FL_SOMENTE_BRL'] = 'S' if qt_nao_brl == 0 else 'N'
    df_brl = df_classificado.filter(F.col('CD_TIP_MOE_CRR') == F.lit('BRL'))
qt_brl_materializado = None
if df_brl is not None:
    df_brl = df_brl.persist(StorageLevel.MEMORY_AND_DISK)
    qt_brl_materializado = df_brl.count()
medir_transformacao('FILTRO_MOEDA_BRL', inicio, qt_brl_materializado, 9)


### V23 — Auditoria do funil de classificação, moeda e participação

Todas as visões auxiliares desta célula têm prefixo `df_aud_` e não retornam ao pipeline funcional.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    if not JANELA_FINANCEIRA_DISPONIVEL:
        print('[AUDITORIA] Funil transacional SKIPPED: janela financeira indisponível.')
        registrar_auditoria('AUD_CLASSIFICACAO_MOEDA', inicio_auditoria, 0, 0, 'SKIPPED')
    else:
        qt_ignoradas_natureza = qt_mov_efetivo - qt_natureza_valida
        df_aud_sem_match = (
            df_natureza_valida.alias('TRAN')
            .join(
                df_categorias.alias('CATEGORIAS'),
                (F.col('TRAN.CD_CTGR_TRAN_OGNL') == F.col('CATEGORIAS.CD_CATEGORIA')) &
                (F.col('TRAN.CD_NTZ_CTB_TRAN') == F.col('CATEGORIAS.TIPO')),
                'left_anti'
            )
        )
        qt_sem_match = df_aud_sem_match.count()
        auditar_painel('FUNIL DE CLASSIFICAÇÃO', [
            ('QT_MOV_RAW', qt_mov_raw),
            ('QT_ANULADAS_C_D', qt_transacoes_removidas),
            ('QT_MOV_EFETIVO', qt_mov_efetivo),
            ('QT_NATUREZA_VALIDA', qt_natureza_valida),
            ('QT_IGNORADAS_NATUREZA', qt_ignoradas_natureza),
            ('QT_CLASSIFICADO', qt_classificado),
            ('QT_SEM_MATCH', qt_sem_match)
        ])
        df_aud_sem_match_pares = df_aud_sem_match.groupBy('CD_CTGR_TRAN_OGNL', 'CD_NTZ_CTB_TRAN').count().withColumnRenamed('count', 'QT')
        qt_pares_sem_match = df_aud_sem_match_pares.count()
        auditar_linhas('Pares sem match exato categoria + natureza', df_aud_sem_match_pares, qt_pares_sem_match)
        df_aud_distribuicao_classificada = df_classificado.groupBy('CD_CLASS_RADAR', 'IN_AGRO', 'IN_PARTICIPA_CALCULO').count().withColumnRenamed('count', 'QT')
        auditar_linhas('Distribuição classificada', df_aud_distribuicao_classificada, df_aud_distribuicao_classificada.count())
        auditar_valores_controlados('df_classificado', df_classificado, ['CD_TIP_MOE_CRR'])
        qt_brl_auditoria = int(qt_brl_materializado or 0)
        qt_descartado_moeda = qt_classificado - qt_brl_auditoria
        df_aud_brl_invalida = df_brl.filter(~F.col('CD_TIP_MOE_CRR').eqNullSafe(F.lit('BRL')))
        auditar_painel('MOEDA E UNIVERSO BRL', [
            ('FL_SOMENTE_BRL', resultado['FL_SOMENTE_BRL']),
            ('QT_CLASSIFICADO', qt_classificado),
            ('QT_BRL', qt_brl_auditoria),
            ('QT_DESCARTADO_MOEDA', qt_descartado_moeda),
            ('QT_BRL_FORA_DO_DOMINIO', df_aud_brl_invalida.count())
        ])
        auditar_valores_controlados('df_brl', df_brl, ['IN_AGRO', 'IN_PARTICIPA_CALCULO'])
        auditar_painel('AGRO E PARTICIPAÇÃO — PRÉ-AGREGAÇÃO', [('FL_TEM_MOV_AGRO', 'AINDA NÃO CALCULADO')])
        df_aud_categorias_observadas = (
            df_brl.groupBy('CD_NTZ_CTB_TRAN', 'CD_CTGR_TRAN_OGNL', 'CD_CLASS_RADAR', 'IN_AGRO', 'IN_PARTICIPA_CALCULO')
            .agg(F.count(F.lit(1)).alias('QT_TRANSACOES'), F.sum('VL_TRAN').alias('SUM_VL_TRAN'))
        )
        auditar_linhas('Categorias observadas no universo BRL classificado', df_aud_categorias_observadas, df_aud_categorias_observadas.count())
        planos_indisponiveis = []
        if AUDITORIA_PLANOS_SPARK:
            for nome_plano, df_aud_plano in [('df_mov_raw', df_mov_raw), ('df_classificado', df_classificado), ('df_brl', df_brl)]:
                print(f'\n[AUDITORIA] PLANO {nome_plano}')
                try:
                    texto_plano = df_aud_plano._jdf.queryExecution().executedPlan().toString()
                    print(texto_plano[:LIMITE_CARACTERES_PLANO])
                    if len(texto_plano) > LIMITE_CARACTERES_PLANO:
                        print(f'[AUDITORIA] plano truncado em {LIMITE_CARACTERES_PLANO} caracteres.')
                except Exception as exc_plano:
                    planos_indisponiveis.append(nome_plano)
                    print(f'[AUDITORIA] plano indisponível neste runtime: {type(exc_plano).__name__}: {exc_plano}')
        else:
            print('[AUDITORIA] Planos Spark desativados por AUDITORIA_PLANOS_SPARK=False.')
        status_auditoria = 'AVISO' if planos_indisponiveis else 'OK'
        detalhe_auditoria = '' if not planos_indisponiveis else f'Planos indisponíveis: {planos_indisponiveis}'
        registrar_auditoria('AUD_CLASSIFICACAO_MOEDA', inicio_auditoria, qt_classificado, len(df_classificado.columns), status_auditoria, detalhe_auditoria)


## Agregações transacionais


In [ ]:
%%spark

inicio = time.perf_counter()
nomes_valores = [
    'VL_TRANS_ENT', 'VL_TRANS_SAI',
    'VL_ENT_REN', 'VL_ENT_EST', 'VL_ENT_RESG', 'VL_ENT_OUT', 'VL_ENT_CRED',
    'VL_SAI_IND', 'VL_SAI_ESS', 'VL_SAI_NAO_ESS', 'VL_SAI_FUT', 'VL_SAI_OBR'
]
if not JANELA_FINANCEIRA_DISPONIVEL:
    resultado['FL_TEM_MOV_AGRO'] = None
    resultado['QT_TRANS_TOTAL'] = None
    resultado['QT_TRANS_ENT'] = None
    resultado['QT_TRANS_SAI'] = None
    for nome in nomes_valores:
        resultado[nome] = None
else:
    zero = F.lit(Decimal('0.00')).cast(DecimalType(25, 2))
    valor = F.coalesce(F.col('VL_TRAN').cast(DecimalType(25, 2)), zero)
    participa = F.col('IN_PARTICIPA_CALCULO') == F.lit('S')
    agg = df_brl.agg(
        F.count(F.lit(1)).cast('long').alias('QT_TRANS_TOTAL'),
        F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == 'C', 1).otherwise(0)).cast('long').alias('QT_TRANS_ENT'),
        F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == 'D', 1).otherwise(0)).cast('long').alias('QT_TRANS_SAI'),
        F.max(F.when(F.col('IN_AGRO') == 'S', 1).otherwise(0)).alias('TEM_AGRO'),
        F.sum(F.when((F.col('CD_NTZ_CTB_TRAN') == 'C') & participa, valor).otherwise(zero)).alias('VL_TRANS_ENT'),
        F.sum(F.when((F.col('CD_NTZ_CTB_TRAN') == 'D') & participa, valor).otherwise(zero)).alias('VL_TRANS_SAI'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 1) & participa, valor).otherwise(zero)).alias('VL_ENT_REN'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 2) & participa, valor).otherwise(zero)).alias('VL_ENT_EST'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 3) & participa, valor).otherwise(zero)).alias('VL_ENT_RESG'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 0) & participa, valor).otherwise(zero)).alias('VL_ENT_OUT'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 4) & participa, valor).otherwise(zero)).alias('VL_ENT_CRED'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 5) & participa, valor).otherwise(zero)).alias('VL_SAI_IND'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 6) & participa, valor).otherwise(zero)).alias('VL_SAI_ESS'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 7) & participa, valor).otherwise(zero)).alias('VL_SAI_NAO_ESS'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 8) & participa, valor).otherwise(zero)).alias('VL_SAI_FUT'),
        F.sum(F.when((F.col('CD_CLASS_RADAR') == 9) & participa, valor).otherwise(zero)).alias('VL_SAI_OBR'),
    ).first().asDict()
    qt_total = int(agg['QT_TRANS_TOTAL'] or 0)
    resultado['FL_TEM_MOV_AGRO'] = 'S' if qt_total > 0 and int(agg['TEM_AGRO'] or 0) == 1 else 'N'
    resultado['QT_TRANS_TOTAL'] = qt_total
    resultado['QT_TRANS_ENT'] = int(agg['QT_TRANS_ENT'] or 0)
    resultado['QT_TRANS_SAI'] = int(agg['QT_TRANS_SAI'] or 0)
    for nome in nomes_valores:
        resultado[nome] = agg[nome] if agg[nome] is not None else Decimal('0.00')

if JANELA_FINANCEIRA_DISPONIVEL:
    resultado['VL_ENT_TOTAL'] = sum((resultado[n] for n in ['VL_ENT_REN','VL_ENT_EST','VL_ENT_RESG','VL_ENT_OUT','VL_ENT_CRED']), Decimal('0.00'))
    resultado['VL_SAI_TOTAL'] = sum((resultado[n] for n in ['VL_SAI_IND','VL_SAI_ESS','VL_SAI_NAO_ESS','VL_SAI_FUT','VL_SAI_OBR']), Decimal('0.00'))
else:
    resultado['VL_ENT_TOTAL'] = None
    resultado['VL_SAI_TOTAL'] = None
metrica_moeda = next(m for m in METRICAS if m['ETAPA_QUERY'] == 'FILTRO_MOEDA_BRL')
metrica_moeda['LINHAS'] = 0 if resultado['QT_TRANS_TOTAL'] is None else resultado['QT_TRANS_TOTAL']
medir_transformacao('AGREGACOES', inicio, 1, 18)


## Resultado orçamentário e percentuais


In [ ]:
%%spark

inicio = time.perf_counter()
Q6 = Decimal('0.000001')
if resultado['VL_ENT_TOTAL'] is None:
    resultado['VL_RES_ORC'] = None
    resultado['PC_SAI_ENT'] = None
    resultado['CD_FAIXA_ORC'] = None
else:
    resultado['VL_RES_ORC'] = resultado['VL_ENT_TOTAL'] - resultado['VL_SAI_TOTAL']
    resultado['PC_SAI_ENT'] = (
        (resultado['VL_SAI_TOTAL'] / resultado['VL_ENT_TOTAL']).quantize(Q6, rounding=ROUND_HALF_UP)
        if resultado['VL_ENT_TOTAL'] > 0 else None
    )
    pc = resultado['PC_SAI_ENT']
    if resultado['QT_TRANS_TOTAL'] == 0 or pc is None:
        resultado['CD_FAIXA_ORC'] = None
    elif Decimal('0.950000') <= pc <= Decimal('1.050000'):
        resultado['CD_FAIXA_ORC'] = 0
    elif Decimal('1.050000') < pc <= Decimal('1.250000'):
        resultado['CD_FAIXA_ORC'] = 1
    elif pc > Decimal('1.250000'):
        resultado['CD_FAIXA_ORC'] = 2
    elif Decimal('0.750000') <= pc < Decimal('0.950000'):
        resultado['CD_FAIXA_ORC'] = 3
    else:
        resultado['CD_FAIXA_ORC'] = 4

faixa = resultado['CD_FAIXA_ORC']
resultado['CD_RES_ORC'] = None if faixa is None else (0 if faixa == 0 else (1 if faixa in (3,4) else 2))
resultado['TX_RES_ORC'] = None if faixa is None else ('Neutro' if faixa == 0 else ('Superavitário' if faixa in (3,4) else 'Deficitário'))
resultado['TX_STS_RES'] = None if faixa in (None,0) else ('Moderado' if faixa in (1,3) else 'Acentuado')
resultado['TX_STS_FINAL'] = {
    0:'Neutro', 1:'Deficitário Moderado', 2:'Deficitário Acentuado',
    3:'Superavitário Moderado', 4:'Superavitário Acentuado'
}.get(faixa)
medir_transformacao('RESULTADO_ORCAMENTARIO', inicio, 1, 7)


In [ ]:
%%spark

inicio = time.perf_counter()
for destino, origem in [
    ('PC_SAI_IND','VL_SAI_IND'), ('PC_SAI_ESS','VL_SAI_ESS'),
    ('PC_SAI_NAO_ESS','VL_SAI_NAO_ESS'), ('PC_SAI_FUT','VL_SAI_FUT'),
    ('PC_SAI_OBR','VL_SAI_OBR')
]:
    renda = resultado['VL_REN_PRES']
    resultado[destino] = (
        (resultado[origem] / renda).quantize(Q6, rounding=ROUND_HALF_UP)
        if renda is not None and renda > 0 and resultado[origem] is not None else None
    )
resultado.update({
    'PC_REF_IND': Decimal('0.750000'),
    'PC_REF_ESS': Decimal('0.500000'),
    'PC_REF_NAO_ESS': Decimal('0.300000'),
    'PC_REF_FUT': Decimal('0.200000'),
    'PC_REF_OBR': Decimal('0.300000'),
})
medir_transformacao('PERCENTUAIS', inicio, 1, 10)


## Pontuações


In [ ]:
%%spark

inicio = time.perf_counter()
qt = resultado['QT_TRANS_TOTAL']
renda = resultado['VL_REN_PRES']
if qt is None or qt == 0 or renda is None:
    for nome in ['IND','ESS','NAO_ESS','FUT','OBR']:
        resultado[f'NR_PONT_CONC_{nome}'] = None
elif renda <= 0:
    for nome in ['IND','ESS','NAO_ESS','FUT','OBR']:
        resultado[f'NR_PONT_CONC_{nome}'] = 0
else:
    p = resultado
    resultado['NR_PONT_CONC_IND'] = 99 if p['PC_SAI_IND'] > Decimal('0.750000') else 0
    resultado['NR_PONT_CONC_ESS'] = 0 if p['PC_SAI_ESS'] < Decimal('0.500000') else (1 if p['PC_SAI_ESS'] < Decimal('0.750000') else 2)
    resultado['NR_PONT_CONC_NAO_ESS'] = 0 if p['PC_SAI_NAO_ESS'] < Decimal('0.300000') else (1 if p['PC_SAI_NAO_ESS'] < Decimal('0.450000') else 2)
    resultado['NR_PONT_CONC_FUT'] = 0 if p['PC_SAI_FUT'] >= Decimal('0.300000') else (1 if p['PC_SAI_FUT'] >= Decimal('0.200000') else 2)
    resultado['NR_PONT_CONC_OBR'] = 0 if p['PC_SAI_OBR'] < Decimal('0.300000') else (1 if p['PC_SAI_OBR'] < Decimal('0.450000') else 2)
medir_transformacao('PONTUACAO_CONCENTRACAO', inicio, 1, 5)


In [ ]:
%%spark

inicio = time.perf_counter()
faixa = resultado['CD_FAIXA_ORC']
resultado['NR_PONT_ORC_IND'] = None if qt is None or qt == 0 else 0
if qt is None or qt == 0 or faixa is None:
    for nome in ['ESS','NAO_ESS','FUT','OBR']:
        resultado[f'NR_PONT_ORC_{nome}'] = None
else:
    regra_ess = 2 if faixa == 2 else (1 if faixa in (0,1) else 0)
    resultado['NR_PONT_ORC_ESS'] = regra_ess
    resultado['NR_PONT_ORC_NAO_ESS'] = regra_ess
    resultado['NR_PONT_ORC_FUT'] = 2 if faixa == 4 else (1 if faixa in (0,3) else 0)
    resultado['NR_PONT_ORC_OBR'] = regra_ess
medir_transformacao('PONTUACAO_ORCAMENTARIA', inicio, 1, 5)


In [ ]:
%%spark

inicio = time.perf_counter()
macro = resultado['CD_MAC_PRFL_CLI']
resultado['NR_PONT_PRFL_IND'] = None if qt is None or qt == 0 else 0
matriz_perfil = {1:(0,1,0,2), 2:(1,0,1,0), 3:(1,0,2,0)}
if macro not in matriz_perfil:
    valores_perfil = (None, None, None, None)
else:
    valores_perfil = matriz_perfil[macro]
for nome, valor in zip(['ESS','NAO_ESS','FUT','OBR'], valores_perfil):
    resultado[f'NR_PONT_PRFL_{nome}'] = valor
medir_transformacao('PONTUACAO_PERFIL', inicio, 1, 5)


In [ ]:
%%spark

inicio = time.perf_counter()
resultado['NR_PONT_IND_FIM'] = resultado['NR_PONT_CONC_IND']
for nome in ['ESS','NAO_ESS','FUT','OBR']:
    parcelas = [
        resultado[f'NR_PONT_CONC_{nome}'],
        resultado[f'NR_PONT_ORC_{nome}'],
        resultado[f'NR_PONT_PRFL_{nome}'],
    ]
    resultado[f'NR_PONT_{nome}_FIM'] = None if any(v is None for v in parcelas) else sum(parcelas)
medir_transformacao('PONTUACAO_FINAL', inicio, 1, 5)


In [ ]:
%%spark

inicio = time.perf_counter()
nomes_finais = ['IND','ESS','NAO_ESS','FUT','OBR']
pontos = [resultado[f'NR_PONT_{n}_FIM'] for n in nomes_finais]
resultado['FL_PONTUACAO_COMPLETA'] = 'S' if all(v is not None for v in pontos) else 'N'
if resultado['FL_PONTUACAO_COMPLETA'] == 'S':
    resultado['NR_PONT_MAX'] = max(pontos)
    resultado['QT_TEMAS_PONT_MAX'] = sum(1 for v in pontos if v == resultado['NR_PONT_MAX'])
    resultado['CD_TEMA_VENCEDOR'] = 9 if resultado['QT_TEMAS_PONT_MAX'] > 1 else pontos.index(resultado['NR_PONT_MAX']) + 1
else:
    resultado['NR_PONT_MAX'] = None
    resultado['QT_TEMAS_PONT_MAX'] = None
    resultado['CD_TEMA_VENCEDOR'] = None
resultado['TX_TEMA_VENCEDOR'] = {
    1:'Categorização dos Gastos', 2:'Gestão de Orçamento',
    3:'Consumo Planejado', 4:'Formação de Reserva',
    5:'Uso Consciente do Crédito', 9:'Empate'
}.get(resultado['CD_TEMA_VENCEDOR'])
medir_transformacao('TEMA_VENCEDOR', inicio, 1, 5)


### V23 — Auditoria de agregações, orçamento, pontuações e tema

A célula apenas apresenta os valores já calculados no dicionário `resultado`; não recalcula nenhuma regra.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    auditar_painel('AGREGAÇÕES TRANSACIONAIS', [
        (nome, resultado[nome]) for nome in [
            'QT_TRANS_TOTAL', 'QT_TRANS_ENT', 'QT_TRANS_SAI',
            'VL_TRANS_ENT', 'VL_TRANS_SAI',
            'VL_ENT_REN', 'VL_ENT_EST', 'VL_ENT_RESG', 'VL_ENT_OUT', 'VL_ENT_CRED', 'VL_ENT_TOTAL',
            'VL_SAI_IND', 'VL_SAI_ESS', 'VL_SAI_NAO_ESS', 'VL_SAI_FUT', 'VL_SAI_OBR', 'VL_SAI_TOTAL'
        ]
    ])
    auditar_painel('RENDA PRESUMIDA E USO NOS PERCENTUAIS', [
        ('DT_REN_PRES_REF', resultado['DT_REN_PRES_REF']),
        ('VL_REN_PRES', resultado['VL_REN_PRES']),
        ('USO', 'percentuais por tema e pontuações de concentração; não compõe as somas transacionais nem o orçamento')
    ])
    for tema in ['IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR']:
        print(f"PC_SAI_{tema} = VL_SAI_{tema} / VL_REN_PRES = {resultado['VL_SAI_' + tema]} / {resultado['VL_REN_PRES']} = {resultado['PC_SAI_' + tema]}")
    invariantes_auditoria = [
        ('QT_TRANS_TOTAL = QT_TRANS_ENT + QT_TRANS_SAI', resultado['QT_TRANS_TOTAL'] == (resultado['QT_TRANS_ENT'] + resultado['QT_TRANS_SAI']) if resultado['QT_TRANS_TOTAL'] is not None else None),
        ('VL_TRANS_ENT = VL_ENT_TOTAL', resultado['VL_TRANS_ENT'] == resultado['VL_ENT_TOTAL'] if resultado['VL_TRANS_ENT'] is not None else None),
        ('VL_TRANS_SAI = VL_SAI_TOTAL', resultado['VL_TRANS_SAI'] == resultado['VL_SAI_TOTAL'] if resultado['VL_TRANS_SAI'] is not None else None)
    ]
    for nome, valor in invariantes_auditoria:
        print(f'[AUDITORIA] {nome}: {"OK" if valor is True else ("NÃO APLICÁVEL" if valor is None else "ERRO")}')
    auditar_painel('ORÇAMENTO: VALORES → PERCENTUAL → FAIXA → RESULTADO', [
        (nome, resultado[nome]) for nome in [
            'VL_ENT_TOTAL', 'VL_SAI_TOTAL', 'VL_RES_ORC', 'PC_SAI_ENT', 'CD_FAIXA_ORC',
            'CD_RES_ORC', 'TX_RES_ORC', 'TX_STS_RES', 'TX_STS_FINAL'
        ]
    ])
    print(f"VL_RES_ORC = VL_ENT_TOTAL - VL_SAI_TOTAL = {resultado['VL_ENT_TOTAL']} - {resultado['VL_SAI_TOTAL']} = {resultado['VL_RES_ORC']}")
    print(f"PC_SAI_ENT = VL_SAI_TOTAL / VL_ENT_TOTAL = {resultado['VL_SAI_TOTAL']} / {resultado['VL_ENT_TOTAL']} = {resultado['PC_SAI_ENT']}")
    print('\nTEMA | PONT_CONC | PONT_ORC | PONT_PRFL | PONT_FINAL')
    for tema in ['IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR']:
        pont_conc = resultado['NR_PONT_CONC_' + tema]
        pont_orc = resultado['NR_PONT_ORC_' + tema]
        pont_prfl = resultado['NR_PONT_PRFL_' + tema]
        pont_final = resultado['NR_PONT_' + tema + '_FIM']
        print(f"{tema} | f(PC_SAI_{tema}={resultado['PC_SAI_' + tema]}, PC_REF_{tema}={resultado['PC_REF_' + tema]}) = {pont_conc} | f(CD_FAIXA_ORC={resultado['CD_FAIXA_ORC']}) = {pont_orc} | f(CD_MAC_PRFL_CLI={resultado['CD_MAC_PRFL_CLI']}) = {pont_prfl} | {pont_final}")
        if tema == 'IND':
            print(f"  NR_PONT_IND_FIM = NR_PONT_CONC_IND = {pont_conc}")
        else:
            print(f"  NR_PONT_{tema}_FIM = NR_PONT_CONC_{tema} + NR_PONT_ORC_{tema} + NR_PONT_PRFL_{tema} = {pont_conc} + {pont_orc} + {pont_prfl} = {pont_final}")
    auditar_painel('TEMA VENCEDOR', [
        (nome, resultado[nome]) for nome in [
            'FL_PONTUACAO_COMPLETA', 'NR_PONT_IND_FIM', 'NR_PONT_ESS_FIM', 'NR_PONT_NAO_ESS_FIM',
            'NR_PONT_FUT_FIM', 'NR_PONT_OBR_FIM', 'NR_PONT_MAX', 'QT_TEMAS_PONT_MAX',
            'CD_TEMA_VENCEDOR', 'TX_TEMA_VENCEDOR'
        ]
    ])
    print(f"NR_PONT_MAX = MAX(NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM) = {resultado['NR_PONT_MAX']}")
    print(f"QT_TEMAS_PONT_MAX = quantidade de temas com pontuação final igual a NR_PONT_MAX = {resultado['QT_TEMAS_PONT_MAX']}")
    print(f"CD_TEMA_VENCEDOR = {resultado['CD_TEMA_VENCEDOR']}; TX_TEMA_VENCEDOR = {resultado['TX_TEMA_VENCEDOR']}")
    if df_brl is None:
        print('\n[AUDITORIA] TRANSAÇÕES DE ENTRADA: NÃO DISPONÍVEIS — janela financeira indisponível.')
    else:
        df_aud_transacoes_entrada = (
            df_brl
            .filter(F.col('CD_NTZ_CTB_TRAN') == F.lit('C'))
            .select(
                'CD_CLI', 'DT_TRAN', 'CD_CTGR_TRAN_OGNL', 'CD_CLASS_RADAR',
                'IN_AGRO', 'IN_PARTICIPA_CALCULO', 'CD_TIP_MOE_CRR', 'VL_TRAN'
            )
            .orderBy(F.col('DT_TRAN').asc_nulls_last())
        )
        qt_transacoes_entrada_auditoria = df_aud_transacoes_entrada.count()
        print(f"\n[AUDITORIA] TODAS AS TRANSAÇÕES DE ENTRADA | QT_AUDITORIA={qt_transacoes_entrada_auditoria} | QT_TRANS_ENT_RESULTADO={resultado['QT_TRANS_ENT']}")
        df_aud_transacoes_entrada.show(qt_transacoes_entrada_auditoria, truncate=False)
    registrar_auditoria('AUD_AGREGACOES_ORCAMENTO_PONTUACOES', inicio_auditoria, 1, len(resultado))


## Schema físico final


In [ ]:
%%spark

SCHEMA_FINAL = StructType([
    StructField('CD_CLI', IntegerType(), False),
    StructField('DT_EXEA', DateType(), False),
    StructField('DT_MES_EXEA', DateType(), False),
    StructField('TS_INCL_TRAN_REF', TimestampType(), False),
    StructField('FL_CPF_UNICO', StringType(), False),
    StructField('CD_CPF', DecimalType(14,0), True),
    StructField('FL_CONTA_ELEGIVEL_UNICA', StringType(), False),
    StructField('TS_DD_INC_MM_CLC_BLC_REF', TimestampType(), True),
    StructField('DD_INC_MM_CLC_BLC', ShortType(), True),
    StructField('DD_INC_MM_CLC_BLC_FALLBACK', ShortType(), True),
    StructField('DT_REN_PRES_REF', DateType(), True),
    StructField('VL_REN_PRES', DecimalType(17,2), True),
    StructField('DT_REF_PRFL', DateType(), True),
    StructField('CD_MAC_PRFL_CLI', IntegerType(), True),
    StructField('NM_MAC_PRFL_CLI', StringType(), True),
    StructField('CD_MIC_PRFL_CLI', IntegerType(), True),
    StructField('NM_MIC_PRFL_CLI', StringType(), True),
    StructField('DT_REF_INI', DateType(), True),
    StructField('DT_REF_FIM', DateType(), True),
    StructField('FL_SOMENTE_BRL', StringType(), True),
    StructField('FL_TEM_MOV_AGRO', StringType(), True),
    StructField('QT_TRANS_TOTAL', LongType(), True),
    StructField('QT_TRANS_ENT', LongType(), True),
    StructField('QT_TRANS_SAI', LongType(), True),
    StructField('VL_TRANS_ENT', DecimalType(25,2), True),
    StructField('VL_TRANS_SAI', DecimalType(25,2), True),
    StructField('VL_ENT_REN', DecimalType(25,2), True),
    StructField('VL_ENT_EST', DecimalType(25,2), True),
    StructField('VL_ENT_RESG', DecimalType(25,2), True),
    StructField('VL_ENT_OUT', DecimalType(25,2), True),
    StructField('VL_ENT_CRED', DecimalType(25,2), True),
    StructField('VL_ENT_TOTAL', DecimalType(25,2), True),
    StructField('VL_SAI_IND', DecimalType(25,2), True),
    StructField('VL_SAI_ESS', DecimalType(25,2), True),
    StructField('VL_SAI_NAO_ESS', DecimalType(25,2), True),
    StructField('VL_SAI_FUT', DecimalType(25,2), True),
    StructField('VL_SAI_OBR', DecimalType(25,2), True),
    StructField('VL_SAI_TOTAL', DecimalType(25,2), True),
    StructField('VL_RES_ORC', DecimalType(25,2), True),
    StructField('PC_SAI_ENT', DecimalType(9,6), True),
    StructField('CD_RES_ORC', IntegerType(), True),
    StructField('TX_RES_ORC', StringType(), True),
    StructField('CD_FAIXA_ORC', IntegerType(), True),
    StructField('TX_STS_RES', StringType(), True),
    StructField('TX_STS_FINAL', StringType(), True),
    StructField('PC_SAI_IND', DecimalType(9,6), True),
    StructField('PC_SAI_ESS', DecimalType(9,6), True),
    StructField('PC_SAI_NAO_ESS', DecimalType(9,6), True),
    StructField('PC_SAI_FUT', DecimalType(9,6), True),
    StructField('PC_SAI_OBR', DecimalType(9,6), True),
    StructField('PC_REF_IND', DecimalType(9,6), False),
    StructField('PC_REF_ESS', DecimalType(9,6), False),
    StructField('PC_REF_NAO_ESS', DecimalType(9,6), False),
    StructField('PC_REF_FUT', DecimalType(9,6), False),
    StructField('PC_REF_OBR', DecimalType(9,6), False),
    StructField('NR_PONT_CONC_IND', IntegerType(), True),
    StructField('NR_PONT_CONC_ESS', IntegerType(), True),
    StructField('NR_PONT_CONC_NAO_ESS', IntegerType(), True),
    StructField('NR_PONT_CONC_FUT', IntegerType(), True),
    StructField('NR_PONT_CONC_OBR', IntegerType(), True),
    StructField('NR_PONT_ORC_IND', IntegerType(), True),
    StructField('NR_PONT_ORC_ESS', IntegerType(), True),
    StructField('NR_PONT_ORC_NAO_ESS', IntegerType(), True),
    StructField('NR_PONT_ORC_FUT', IntegerType(), True),
    StructField('NR_PONT_ORC_OBR', IntegerType(), True),
    StructField('NR_PONT_PRFL_IND', IntegerType(), True),
    StructField('NR_PONT_PRFL_ESS', IntegerType(), True),
    StructField('NR_PONT_PRFL_NAO_ESS', IntegerType(), True),
    StructField('NR_PONT_PRFL_FUT', IntegerType(), True),
    StructField('NR_PONT_PRFL_OBR', IntegerType(), True),
    StructField('NR_PONT_IND_FIM', IntegerType(), True),
    StructField('NR_PONT_ESS_FIM', IntegerType(), True),
    StructField('NR_PONT_NAO_ESS_FIM', IntegerType(), True),
    StructField('NR_PONT_FUT_FIM', IntegerType(), True),
    StructField('NR_PONT_OBR_FIM', IntegerType(), True),
    StructField('FL_PONTUACAO_COMPLETA', StringType(), False),
    StructField('NR_PONT_MAX', IntegerType(), True),
    StructField('QT_TEMAS_PONT_MAX', IntegerType(), True),
    StructField('CD_TEMA_VENCEDOR', IntegerType(), True),
    StructField('TX_TEMA_VENCEDOR', StringType(), True),
])
if len(SCHEMA_FINAL.fields) != 80:
    raise AssertionError('O schema final deve possuir exatamente 80 colunas.')


In [ ]:
%%spark

inicio = time.perf_counter()
colunas_finais = [campo.name for campo in SCHEMA_FINAL.fields]
ausentes = [nome for nome in colunas_finais if nome not in resultado]
extras = [nome for nome in resultado if nome not in colunas_finais]
if ausentes or extras:
    raise AssertionError(f'Contrato físico divergente. Ausentes={ausentes}; extras={extras}')
df_final = spark.createDataFrame(
    [tuple(resultado[nome] for nome in colunas_finais)],
    schema=SCHEMA_FINAL,
    verifySchema=True,
).persist(StorageLevel.MEMORY_AND_DISK)
qt_linhas_final = df_final.count()
linha_final = df_final.first()
if qt_linhas_final != 1:
    raise AssertionError(f'O resultado final deve possuir exatamente uma linha; encontrado={qt_linhas_final}.')
medir_transformacao('MONTAGEM_SCHEMA_FINAL', inicio, qt_linhas_final, len(df_final.columns))


## Validações contratuais e publicação temporária


In [ ]:
%%spark

inicio = time.perf_counter()
erros = []
if len(df_final.columns) != 80:
    erros.append('Quantidade de colunas diferente de 80.')
if df_final.schema != SCHEMA_FINAL:
    erros.append('Schema físico divergente.')
if linha_final['CD_CLI'] != CD_CLI:
    erros.append('CD_CLI final divergente.')
if linha_final['FL_CPF_UNICO'] == 'N' and any(linha_final[n] is not None for n in ['CD_CPF','DT_REN_PRES_REF','VL_REN_PRES']):
    erros.append('Nulabilidade de CPF/renda inválida.')
if linha_final['FL_CONTA_ELEGIVEL_UNICA'] == 'N' and linha_final['DD_INC_MM_CLC_BLC_FALLBACK'] is not None:
    erros.append('Fallback de conta inválido.')
for nome in ['FL_CPF_UNICO','FL_CONTA_ELEGIVEL_UNICA','FL_SOMENTE_BRL','FL_TEM_MOV_AGRO','FL_PONTUACAO_COMPLETA']:
    if linha_final[nome] not in ('S','N',None):
        erros.append(f'Domínio S/N inválido em {nome}.')
if JANELA_FINANCEIRA_DISPONIVEL:
    if linha_final['QT_TRANS_TOTAL'] != linha_final['QT_TRANS_ENT'] + linha_final['QT_TRANS_SAI']:
        erros.append('QT_TRANS_TOTAL != QT_TRANS_ENT + QT_TRANS_SAI.')
    if linha_final['VL_TRANS_ENT'] != linha_final['VL_ENT_TOTAL']:
        erros.append('VL_TRANS_ENT != VL_ENT_TOTAL.')
    if linha_final['VL_TRANS_SAI'] != linha_final['VL_SAI_TOTAL']:
        erros.append('VL_TRANS_SAI != VL_SAI_TOTAL.')
if linha_final['FL_PONTUACAO_COMPLETA'] == 'N' and any(linha_final[n] is not None for n in ['NR_PONT_MAX','QT_TEMAS_PONT_MAX','CD_TEMA_VENCEDOR','TX_TEMA_VENCEDOR']):
    erros.append('Tema deveria ser nulo com pontuação incompleta.')
if linha_final['QT_TEMAS_PONT_MAX'] is not None and linha_final['QT_TEMAS_PONT_MAX'] > 1:
    if linha_final['CD_TEMA_VENCEDOR'] != 9 or linha_final['TX_TEMA_VENCEDOR'] != 'Empate':
        erros.append('Empate final inconsistente.')
if qt_linhas_final != 1:
    erros.append('Cardinalidade final diferente de uma linha.')

# CPF e renda.
fl_cpf_esperada = 'S' if int(cpf['QT_CPF']) == 1 else 'N'
if linha_final['FL_CPF_UNICO'] != fl_cpf_esperada:
    erros.append('FL_CPF_UNICO divergente da contagem distinta.')
if fl_cpf_esperada == 'S' and linha_final['CD_CPF'] is None:
    erros.append('CPF único não foi preservado.')
if fl_cpf_esperada == 'N' and any(linha_final[n] is not None for n in ['CD_CPF','DT_REN_PRES_REF','VL_REN_PRES']):
    erros.append('Cliente sem CPF único possui CPF/renda preenchidos.')
if linha_renda is None:
    if linha_final['DT_REN_PRES_REF'] is not None or linha_final['VL_REN_PRES'] is not None:
        erros.append('Renda preenchida sem linha selecionada.')
elif linha_final['DT_REN_PRES_REF'] != linha_renda['DT_INCL_REN_AVLD'] or linha_final['VL_REN_PRES'] != linha_renda['VL_REN']:
    erros.append('Data e valor de renda não vieram da mesma linha selecionada.')

# Conta, ciclo e janela.
fl_conta_esperada = 'S' if len(contas) == 1 else 'N'
if linha_final['FL_CONTA_ELEGIVEL_UNICA'] != fl_conta_esperada:
    erros.append('FL_CONTA_ELEGIVEL_UNICA divergente.')
if fl_conta_esperada == 'N':
    if any(linha_final[n] is not None for n in ['TS_DD_INC_MM_CLC_BLC_REF','DD_INC_MM_CLC_BLC','DD_INC_MM_CLC_BLC_FALLBACK','DT_REF_INI','DT_REF_FIM']):
        erros.append('Cliente sem conta única possui ciclo/janela.')
elif linha_ciclo is None:
    if linha_final['TS_DD_INC_MM_CLC_BLC_REF'] is not None or linha_final['DD_INC_MM_CLC_BLC'] is not None or linha_final['DD_INC_MM_CLC_BLC_FALLBACK'] != 1:
        erros.append('Fallback de ciclo incorreto.')
elif linha_final['TS_DD_INC_MM_CLC_BLC_REF'] != linha_ciclo['TS_ULT_EXEA_PSQ'] or linha_final['DD_INC_MM_CLC_BLC'] != linha_ciclo['DD_INC_MM_CLC_BLC']:
    erros.append('Timestamp e dia do ciclo não vieram da mesma linha.')
if (linha_final['DT_REF_INI'] is None) != (linha_final['DT_REF_FIM'] is None):
    erros.append('Janela financeira parcial.')
if linha_final['DT_REF_INI'] is not None and linha_final['DT_REF_INI'] > linha_final['DT_REF_FIM']:
    erros.append('Janela financeira invertida.')

# Perfil selecionado na mesma linha.
if linha_perfil is None:
    if any(linha_final[n] is not None for n in ['DT_REF_PRFL','CD_MAC_PRFL_CLI','NM_MAC_PRFL_CLI','CD_MIC_PRFL_CLI','NM_MIC_PRFL_CLI']):
        erros.append('Perfil preenchido sem linha selecionada.')
else:
    perfil_esperado = [linha_perfil['DT_REF'], linha_perfil['CD_MAC_PRFL_CLI'], linha_perfil['NM_MAC_PRFL_CLI'], linha_perfil['CD_MIC_PRFL_CLI'], linha_perfil['NM_MIC_PRFL_CLI']]
    perfil_final = [linha_final['DT_REF_PRFL'], linha_final['CD_MAC_PRFL_CLI'], linha_final['NM_MAC_PRFL_CLI'], linha_final['CD_MIC_PRFL_CLI'], linha_final['NM_MIC_PRFL_CLI']]
    if perfil_final != perfil_esperado:
        erros.append('Atributos de perfil não vieram da mesma linha.')

# Moeda pré-BRL e universo BRL.
if not JANELA_FINANCEIRA_DISPONIVEL:
    if linha_final['FL_SOMENTE_BRL'] is not None or linha_final['FL_TEM_MOV_AGRO'] is not None or linha_final['QT_TRANS_TOTAL'] is not None:
        erros.append('Atributos transacionais preenchidos sem janela.')
else:
    fl_brl_esperada = 'N' if qt_classificado == 0 else ('S' if qt_nao_brl == 0 else 'N')
    if linha_final['FL_SOMENTE_BRL'] != fl_brl_esperada:
        erros.append('FL_SOMENTE_BRL divergente do universo classificado.')
    qt_brl = int(agg['QT_TRANS_TOTAL'] or 0)
    if linha_final['QT_TRANS_TOTAL'] != qt_brl:
        erros.append('Volumetria divergente do universo BRL.')
    fl_agro_esperada = 'S' if qt_brl > 0 and int(agg['TEM_AGRO'] or 0) == 1 else 'N'
    if linha_final['FL_TEM_MOV_AGRO'] != fl_agro_esperada:
        erros.append('FL_TEM_MOV_AGRO divergente do universo BRL.')

# Orçamento e percentuais.
if linha_final['VL_ENT_TOTAL'] is not None:
    if linha_final['VL_RES_ORC'] != linha_final['VL_ENT_TOTAL'] - linha_final['VL_SAI_TOTAL']:
        erros.append('VL_RES_ORC divergente.')
    pc_orc_esperado = ((linha_final['VL_SAI_TOTAL'] / linha_final['VL_ENT_TOTAL']).quantize(Q6, rounding=ROUND_HALF_UP) if linha_final['VL_ENT_TOTAL'] > 0 else None)
    if linha_final['PC_SAI_ENT'] != pc_orc_esperado:
        erros.append('PC_SAI_ENT divergente.')
faixa_final = linha_final['CD_FAIXA_ORC']
if linha_final['QT_TRANS_TOTAL'] in (None, 0) or linha_final['PC_SAI_ENT'] is None:
    faixa_esperada = None
elif Decimal('0.950000') <= linha_final['PC_SAI_ENT'] <= Decimal('1.050000'):
    faixa_esperada = 0
elif Decimal('1.050000') < linha_final['PC_SAI_ENT'] <= Decimal('1.250000'):
    faixa_esperada = 1
elif linha_final['PC_SAI_ENT'] > Decimal('1.250000'):
    faixa_esperada = 2
elif Decimal('0.750000') <= linha_final['PC_SAI_ENT'] < Decimal('0.950000'):
    faixa_esperada = 3
else:
    faixa_esperada = 4
if linha_final['CD_FAIXA_ORC'] != faixa_esperada:
    erros.append('CD_FAIXA_ORC divergente de PC_SAI_ENT.')

cd_res_esperado = None if faixa_final is None else (0 if faixa_final == 0 else (1 if faixa_final in (3,4) else 2))
if linha_final['CD_RES_ORC'] != cd_res_esperado:
    erros.append('CD_RES_ORC divergente.')
tx_res_esperado = None if faixa_final is None else ('Neutro' if faixa_final == 0 else ('Superavitário' if faixa_final in (3,4) else 'Deficitário'))
tx_sts_esperado = None if faixa_final in (None,0) else ('Moderado' if faixa_final in (1,3) else 'Acentuado')
tx_final_esperado = {0:'Neutro',1:'Deficitário Moderado',2:'Deficitário Acentuado',3:'Superavitário Moderado',4:'Superavitário Acentuado'}.get(faixa_final)
if (linha_final['TX_RES_ORC'], linha_final['TX_STS_RES'], linha_final['TX_STS_FINAL']) != (tx_res_esperado, tx_sts_esperado, tx_final_esperado):
    erros.append('Textos orçamentários divergentes de CD_FAIXA_ORC.')
referencias_esperadas = {'PC_REF_IND':Decimal('0.750000'),'PC_REF_ESS':Decimal('0.500000'),'PC_REF_NAO_ESS':Decimal('0.300000'),'PC_REF_FUT':Decimal('0.200000'),'PC_REF_OBR':Decimal('0.300000')}
for nome, esperado in referencias_esperadas.items():
    if linha_final[nome] != esperado:
        erros.append(f'Constante de referência divergente em {nome}.')
for destino, origem in [('PC_SAI_IND','VL_SAI_IND'),('PC_SAI_ESS','VL_SAI_ESS'),('PC_SAI_NAO_ESS','VL_SAI_NAO_ESS'),('PC_SAI_FUT','VL_SAI_FUT'),('PC_SAI_OBR','VL_SAI_OBR')]:
    renda_final = linha_final['VL_REN_PRES']
    pc_esperado = ((linha_final[origem] / renda_final).quantize(Q6, rounding=ROUND_HALF_UP) if renda_final is not None and renda_final > 0 and linha_final[origem] is not None else None)
    if linha_final[destino] != pc_esperado:
        erros.append(f'{destino} divergente.')

# Pontuações de concentração.
if linha_final['QT_TRANS_TOTAL'] is None or linha_final['QT_TRANS_TOTAL'] == 0 or linha_final['VL_REN_PRES'] is None:
    conc_esperada = {n: None for n in ['IND','ESS','NAO_ESS','FUT','OBR']}
elif linha_final['VL_REN_PRES'] <= 0:
    conc_esperada = {n: 0 for n in ['IND','ESS','NAO_ESS','FUT','OBR']}
else:
    conc_esperada = {
        'IND': 99 if linha_final['PC_SAI_IND'] > Decimal('0.750000') else 0,
        'ESS': 0 if linha_final['PC_SAI_ESS'] < Decimal('0.500000') else (1 if linha_final['PC_SAI_ESS'] < Decimal('0.750000') else 2),
        'NAO_ESS': 0 if linha_final['PC_SAI_NAO_ESS'] < Decimal('0.300000') else (1 if linha_final['PC_SAI_NAO_ESS'] < Decimal('0.450000') else 2),
        'FUT': 0 if linha_final['PC_SAI_FUT'] >= Decimal('0.300000') else (1 if linha_final['PC_SAI_FUT'] >= Decimal('0.200000') else 2),
        'OBR': 0 if linha_final['PC_SAI_OBR'] < Decimal('0.300000') else (1 if linha_final['PC_SAI_OBR'] < Decimal('0.450000') else 2),
    }
for nome, esperado in conc_esperada.items():
    if linha_final[f'NR_PONT_CONC_{nome}'] != esperado:
        erros.append(f'Pontuação de concentração divergente para {nome}.')

# Pontuações orçamentárias.
qt_final = linha_final['QT_TRANS_TOTAL']
faixa_final = linha_final['CD_FAIXA_ORC']
if linha_final['NR_PONT_ORC_IND'] != (None if qt_final is None or qt_final == 0 else 0):
    erros.append('NR_PONT_ORC_IND divergente.')
if qt_final is None or qt_final == 0 or faixa_final is None:
    orc_esperada = {n: None for n in ['ESS','NAO_ESS','FUT','OBR']}
else:
    base_ess = 2 if faixa_final == 2 else (1 if faixa_final in (0,1) else 0)
    orc_esperada = {'ESS':base_ess, 'NAO_ESS':base_ess, 'FUT':2 if faixa_final == 4 else (1 if faixa_final in (0,3) else 0), 'OBR':base_ess}
for nome, esperado in orc_esperada.items():
    if linha_final[f'NR_PONT_ORC_{nome}'] != esperado:
        erros.append(f'Pontuação orçamentária divergente para {nome}.')

# Pontuações de perfil.
if linha_final['NR_PONT_PRFL_IND'] != (None if qt_final is None or qt_final == 0 else 0):
    erros.append('NR_PONT_PRFL_IND divergente.')
perfil_matriz = {1:(0,1,0,2), 2:(1,0,1,0), 3:(1,0,2,0)}
perfil_esperado_pontos = perfil_matriz.get(linha_final['CD_MAC_PRFL_CLI'], (None,None,None,None))
for nome, esperado in zip(['ESS','NAO_ESS','FUT','OBR'], perfil_esperado_pontos):
    if linha_final[f'NR_PONT_PRFL_{nome}'] != esperado:
        erros.append(f'Pontuação de perfil divergente para {nome}.')

# Pontuações finais e tema.
if linha_final['NR_PONT_IND_FIM'] != linha_final['NR_PONT_CONC_IND']:
    erros.append('NR_PONT_IND_FIM divergente.')
for nome in ['ESS','NAO_ESS','FUT','OBR']:
    partes = [linha_final[f'NR_PONT_CONC_{nome}'], linha_final[f'NR_PONT_ORC_{nome}'], linha_final[f'NR_PONT_PRFL_{nome}']]
    pont_esperada = None if any(v is None for v in partes) else sum(partes)
    if linha_final[f'NR_PONT_{nome}_FIM'] != pont_esperada:
        erros.append(f'Pontuação final divergente para {nome}.')
pontos_finais = [linha_final[f'NR_PONT_{n}_FIM'] for n in ['IND','ESS','NAO_ESS','FUT','OBR']]
fl_completa_esperada = 'S' if all(v is not None for v in pontos_finais) else 'N'
if linha_final['FL_PONTUACAO_COMPLETA'] != fl_completa_esperada:
    erros.append('FL_PONTUACAO_COMPLETA divergente.')
if fl_completa_esperada == 'S':
    max_esperado = max(pontos_finais)
    qt_max_esperada = sum(1 for v in pontos_finais if v == max_esperado)
    tema_esperado = 9 if qt_max_esperada > 1 else pontos_finais.index(max_esperado) + 1
    if (linha_final['NR_PONT_MAX'], linha_final['QT_TEMAS_PONT_MAX'], linha_final['CD_TEMA_VENCEDOR']) != (max_esperado, qt_max_esperada, tema_esperado):
        erros.append('Seleção do tema vencedor divergente.')
tx_tema_esperado = {1:'Categorização dos Gastos',2:'Gestão de Orçamento',3:'Consumo Planejado',4:'Formação de Reserva',5:'Uso Consciente do Crédito',9:'Empate'}.get(linha_final['CD_TEMA_VENCEDOR'])
if linha_final['TX_TEMA_VENCEDOR'] != tx_tema_esperado:
    erros.append('TX_TEMA_VENCEDOR divergente do código vencedor.')

if erros:
    STATUS_CONTRATO_FINAL = 'DIVERGENTE'
    remover_temp_view_resultado()
    raise AssertionError(' | '.join(erros))
STATUS_CONTRATO_FINAL = 'OK'
medir_validacao('VALIDACOES_FINAIS', inicio, 1, 80)
print(f'[V23][GATE] STATUS_CONTRATO_FINAL={STATUS_CONTRATO_FINAL}')


### V23 — Auditoria vertical do contrato final

Apresentação posterior à validação contratual congelada. Não modifica `SCHEMA_FINAL`, `df_final` ou a view.


In [ ]:
%%spark

if AUDITORIA_ATIVA:
    inicio_auditoria = time.perf_counter()
    print('ORDEM | COLUNA | TIPO_ESPERADO | TIPO_OBTIDO | NULLABLE_ESPERADO | VALOR | STATUS')
    for ordem, campo_esperado in enumerate(SCHEMA_FINAL.fields, start=1):
        campo_obtido = df_final.schema[campo_esperado.name]
        valor = linha_final[campo_esperado.name]
        status = 'OK'
        if campo_obtido.dataType != campo_esperado.dataType:
            status = 'ERRO_TIPO'
        elif campo_obtido.nullable != campo_esperado.nullable:
            status = 'ERRO_NULABILIDADE'
        elif campo_esperado.name in ['FL_CPF_UNICO','FL_CONTA_ELEGIVEL_UNICA','FL_SOMENTE_BRL','FL_TEM_MOV_AGRO','FL_PONTUACAO_COMPLETA'] and valor not in ('S', 'N', None):
            status = 'ERRO_DOMINIO'
        print(f'{ordem} | {campo_esperado.name} | {campo_esperado.dataType.simpleString()} | {campo_obtido.dataType.simpleString()} | {campo_esperado.nullable} | {valor} | {status}')
    for nome_df in ['df_cliente_raw', 'df_ciclo_raw', 'df_renda_raw', 'df_perfil_raw', 'df_mov_raw', 'df_mov_efetivo', 'df_classificado', 'df_brl', 'df_final']:
        df_aud_schema = globals().get(nome_df)
        if df_aud_schema is not None:
            auditar_schema(nome_df, df_aud_schema)
    registrar_auditoria('AUD_CONTRATO_FINAL_VERTICAL', inicio_auditoria, 1, 80)


## V23 — Suíte sintética bloqueante da anulação

Executada exclusivamente em Spark/memória, sem consulta externa e sem reutilizar ou alterar DataFrames do cliente real. O estado `STATUS_TESTES_SINTETICOS` pertence somente à execução corrente.


In [ ]:
%%spark

# V23 — SUÍTE SINTÉTICA ISOLADA
# Todos os DataFrames e variáveis locais da suíte usam prefixos df_teste_/teste_.
STATUS_TESTES_SINTETICOS = 'NOT_EXECUTED'
teste_inicio = time.perf_counter()
teste_resultados = []
teste_chave_anulacao = ['CD_CLI', 'DT_TRAN', 'VL_TRAN', 'CD_TIP_MOE_CRR']
teste_colunas_funcionais = [
    'CD_CLI', 'DT_TRAN', 'CD_NTZ_CTB_TRAN',
    'CD_CTGR_TRAN_OGNL', 'CD_TIP_MOE_CRR', 'VL_TRAN'
]
teste_schema_raw = SCHEMA_Q5_SKIPPED

def teste_assert(teste_condicao, teste_mensagem):
    if not teste_condicao:
        raise AssertionError(teste_mensagem)

def teste_criar_df(teste_linhas):
    return spark.createDataFrame(teste_linhas, teste_schema_raw)

def teste_obter_chaves(df_teste_raw):
    teste_elegivel = (
        F.col('CD_NTZ_CTB_TRAN').isin('C', 'D') &
        F.col('DT_TRAN').isNotNull() &
        F.col('VL_TRAN').isNotNull() &
        F.col('CD_TIP_MOE_CRR').isNotNull()
    )
    return (
        df_teste_raw
        .filter(teste_elegivel)
        .groupBy(*teste_chave_anulacao)
        .agg(
            F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == F.lit('C'), 1).otherwise(0)).cast('long').alias('QT_C'),
            F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == F.lit('D'), 1).otherwise(0)).cast('long').alias('QT_D')
        )
        .withColumn('QT_PARES_ANULADOS', F.least(F.col('QT_C'), F.col('QT_D')).cast('long'))
        .filter(F.col('QT_PARES_ANULADOS') > F.lit(0))
    )

def teste_contar_pares_residuais(df_teste_efetivo, df_teste_chaves):
    if df_teste_chaves is None:
        return 0
    teste_elegivel = (
        F.col('CD_NTZ_CTB_TRAN').isin('C', 'D') &
        F.col('DT_TRAN').isNotNull() &
        F.col('VL_TRAN').isNotNull() &
        F.col('CD_TIP_MOE_CRR').isNotNull()
    )
    df_teste_residuais = (
        df_teste_efetivo
        .filter(teste_elegivel)
        .join(df_teste_chaves.select(*teste_chave_anulacao), teste_chave_anulacao, 'inner')
        .groupBy(*teste_chave_anulacao)
        .agg(
            F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == F.lit('C'), 1).otherwise(0)).alias('QT_C'),
            F.sum(F.when(F.col('CD_NTZ_CTB_TRAN') == F.lit('D'), 1).otherwise(0)).alias('QT_D')
        )
        .withColumn('QT_PARES_RESIDUAIS', F.least(F.col('QT_C'), F.col('QT_D')))
        .filter(F.col('QT_PARES_RESIDUAIS') > F.lit(0))
    )
    return int(df_teste_residuais.count())

def teste_exigir_sem_residuais(df_teste_efetivo, df_teste_chaves):
    teste_qt_residuais = teste_contar_pares_residuais(df_teste_efetivo, df_teste_chaves)
    if teste_qt_residuais != 0:
        raise AssertionError(
            'Gate pós-anulação falhou: '
            f'QT_CHAVES_COM_PARES_RESIDUAIS={teste_qt_residuais}.'
        )
    return teste_qt_residuais

def teste_aplicar_anulacao(df_teste_raw):
    teste_qt_raw = int(df_teste_raw.count())
    df_teste_chaves = teste_obter_chaves(df_teste_raw)
    teste_resumo = df_teste_chaves.agg(
        F.count(F.lit(1)).cast('long').alias('QT_CHAVES'),
        F.sum('QT_PARES_ANULADOS').cast('long').alias('QT_PARES')
    ).first()
    teste_qt_chaves = int(teste_resumo['QT_CHAVES'] or 0)
    teste_qt_pares = int(teste_resumo['QT_PARES'] or 0)

    if teste_qt_pares == 0:
        # O fast path retorna somente uma projeção do raw e não persiste nada.
        df_teste_efetivo = df_teste_raw.select(*teste_colunas_funcionais)
        df_teste_chaves_ativas = None
        teste_ids_removidos = []
        teste_fast_path = True
    else:
        teste_janela = Window.partitionBy(
            *(teste_chave_anulacao + ['CD_NTZ_CTB_TRAN'])
        ).orderBy(F.col('NR_TRAN_INST_PCT').asc())
        df_teste_marcado = (
            df_teste_raw
            .join(
                df_teste_chaves.select(*teste_chave_anulacao, 'QT_PARES_ANULADOS'),
                teste_chave_anulacao,
                'left'
            )
            .withColumn('_RN_ANULACAO', F.row_number().over(teste_janela))
            .withColumn(
                '_FL_ANULADA',
                F.when(
                    F.col('QT_PARES_ANULADOS').isNotNull() &
                    F.col('CD_NTZ_CTB_TRAN').isin('C', 'D') &
                    (F.col('_RN_ANULACAO') <= F.col('QT_PARES_ANULADOS')),
                    F.lit(True)
                ).otherwise(F.lit(False))
            )
        )
        teste_ids_removidos = sorted(
            int(teste_linha['NR_TRAN_INST_PCT'])
            for teste_linha in df_teste_marcado.filter(F.col('_FL_ANULADA')).select('NR_TRAN_INST_PCT').collect()
        )
        df_teste_efetivo = df_teste_marcado.filter(~F.col('_FL_ANULADA')).select(*teste_colunas_funcionais)
        df_teste_chaves_ativas = df_teste_chaves
        teste_fast_path = False

    teste_qt_efetivo = int(df_teste_efetivo.count())
    teste_qt_removidas = teste_qt_raw - teste_qt_efetivo
    teste_assert(teste_qt_removidas == 2 * teste_qt_pares, 'Invariante sintética removidas != 2*pares.')
    teste_assert(teste_qt_efetivo == teste_qt_raw - teste_qt_removidas, 'Invariante sintética de cardinalidade falhou.')
    teste_assert(df_teste_efetivo.columns == teste_colunas_funcionais, 'Schema funcional sintético divergente.')
    teste_qt_residuais = teste_exigir_sem_residuais(df_teste_efetivo, df_teste_chaves_ativas)

    return {
        'df_teste_efetivo': df_teste_efetivo,
        'teste_qt_raw': teste_qt_raw,
        'teste_qt_chaves': teste_qt_chaves,
        'teste_qt_pares': teste_qt_pares,
        'teste_qt_removidas': teste_qt_removidas,
        'teste_qt_efetivo': teste_qt_efetivo,
        'teste_qt_residuais': teste_qt_residuais,
        'teste_ids_removidos': teste_ids_removidos,
        'teste_fast_path': teste_fast_path,
    }

def teste_registrar(teste_nome, teste_funcao):
    teste_funcao()
    teste_resultados.append((teste_nome, 'OK'))

try:
    teste_data_1 = datetime.date(2026, 1, 10)
    teste_data_2 = datetime.date(2026, 1, 11)

    def teste_cenario_zero_pares():
        df_teste_raw = teste_criar_df([
            (1, 1, teste_data_1, 'C', 1, 'BRL', Decimal('10.00')),
            (2, 1, teste_data_1, 'D', 5, 'BRL', Decimal('20.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        df_teste_efetivo = teste_saida['df_teste_efetivo']
        teste_assert(teste_saida['teste_fast_path'], 'Zero pares não utilizou fast path.')
        teste_assert(teste_saida['teste_qt_pares'] == 0, 'Zero pares detectou par indevido.')
        teste_assert(teste_saida['teste_qt_efetivo'] == teste_saida['teste_qt_raw'] == 2, 'Fast path alterou cardinalidade.')
        teste_assert(len(df_teste_efetivo.columns) == 6, 'Fast path não preservou seis colunas funcionais.')
        teste_plano = df_teste_efetivo._jdf.queryExecution().optimizedPlan().toString()
        teste_plano_lower = teste_plano.lower()
        for teste_token in ['window', 'row_number', 'sort', 'join']:
            teste_assert(teste_token not in teste_plano_lower, f'Fast path contém operação proibida no plano: {teste_token}.')
        teste_storage = df_teste_efetivo.storageLevel
        teste_assert(not teste_storage.useMemory and not teste_storage.useDisk, 'Fast path criou segunda persistência.')

    def teste_cenario_1c1d():
        df_teste_raw = teste_criar_df([
            (11, 1, teste_data_1, 'C', 1, 'BRL', Decimal('100.00')),
            (12, 1, teste_data_1, 'D', 5, 'BRL', Decimal('100.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_assert(teste_saida['teste_qt_pares'] == 1 and teste_saida['teste_qt_efetivo'] == 0, '1C+1D divergente.')
        teste_assert(teste_saida['teste_ids_removidos'] == [11, 12], 'IDs 1C+1D divergentes.')

    def teste_cenario_2c1d():
        df_teste_raw = teste_criar_df([
            (20, 1, teste_data_1, 'C', 1, 'BRL', Decimal('200.00')),
            (10, 1, teste_data_1, 'C', 1, 'BRL', Decimal('200.00')),
            (30, 1, teste_data_1, 'D', 5, 'BRL', Decimal('200.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_linhas = teste_saida['df_teste_efetivo'].select('CD_NTZ_CTB_TRAN').collect()
        teste_assert(teste_saida['teste_ids_removidos'] == [10, 30], 'Determinismo 2C+1D divergente.')
        teste_assert(len(teste_linhas) == 1 and teste_linhas[0]['CD_NTZ_CTB_TRAN'] == 'C', 'Excedente C não foi preservado.')

    def teste_cenario_1c2d():
        df_teste_raw = teste_criar_df([
            (10, 1, teste_data_1, 'C', 1, 'BRL', Decimal('300.00')),
            (30, 1, teste_data_1, 'D', 5, 'BRL', Decimal('300.00')),
            (20, 1, teste_data_1, 'D', 5, 'BRL', Decimal('300.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_linhas = teste_saida['df_teste_efetivo'].select('CD_NTZ_CTB_TRAN').collect()
        teste_assert(teste_saida['teste_ids_removidos'] == [10, 20], 'Determinismo 1C+2D divergente.')
        teste_assert(len(teste_linhas) == 1 and teste_linhas[0]['CD_NTZ_CTB_TRAN'] == 'D', 'Excedente D não foi preservado.')

    def teste_cenario_multiplos_pares():
        df_teste_raw = teste_criar_df([
            (10, 1, teste_data_1, 'C', 1, 'BRL', Decimal('400.00')),
            (20, 1, teste_data_1, 'C', 1, 'BRL', Decimal('400.00')),
            (30, 1, teste_data_1, 'C', 1, 'BRL', Decimal('400.00')),
            (40, 1, teste_data_1, 'D', 5, 'BRL', Decimal('400.00')),
            (50, 1, teste_data_1, 'D', 5, 'BRL', Decimal('400.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_assert(teste_saida['teste_qt_pares'] == 2 and teste_saida['teste_qt_efetivo'] == 1, 'Múltiplos pares divergentes.')
        teste_assert(teste_saida['teste_ids_removidos'] == [10, 20, 40, 50], 'IDs de múltiplos pares divergentes.')

    def teste_cenario_moeda_diferente():
        df_teste_raw = teste_criar_df([
            (1, 1, teste_data_1, 'C', 1, 'BRL', Decimal('500.00')),
            (2, 1, teste_data_1, 'D', 5, 'USD', Decimal('500.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_assert(teste_saida['teste_qt_pares'] == 0 and teste_saida['teste_qt_efetivo'] == 2, 'Moedas diferentes cancelaram indevidamente.')

    def teste_cenario_moeda_vazia():
        df_teste_raw = teste_criar_df([
            (1, 1, teste_data_1, 'C', 1, '', Decimal('600.00')),
            (2, 1, teste_data_1, 'D', 5, '', Decimal('600.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_assert(teste_saida['teste_qt_pares'] == 1 and teste_saida['teste_qt_efetivo'] == 0, 'Moeda vazia idêntica deveria formar par.')

    def teste_cenario_moeda_nula():
        df_teste_raw = teste_criar_df([
            (1, 1, teste_data_1, 'C', 1, None, Decimal('700.00')),
            (2, 1, teste_data_1, 'D', 5, None, Decimal('700.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_assert(teste_saida['teste_qt_pares'] == 0 and teste_saida['teste_qt_efetivo'] == 2, 'Moeda NULL participou indevidamente.')

    def teste_cenario_valor_nulo():
        df_teste_raw = teste_criar_df([
            (1, 1, teste_data_1, 'C', 1, 'BRL', None),
            (2, 1, teste_data_1, 'D', 5, 'BRL', None),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_assert(teste_saida['teste_qt_pares'] == 0 and teste_saida['teste_qt_efetivo'] == 2, 'Valor NULL participou indevidamente.')

    def teste_cenario_datas_diferentes():
        df_teste_raw = teste_criar_df([
            (1, 1, teste_data_1, 'C', 1, 'BRL', Decimal('800.00')),
            (2, 1, teste_data_2, 'D', 5, 'BRL', Decimal('800.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_assert(teste_saida['teste_qt_pares'] == 0 and teste_saida['teste_qt_efetivo'] == 2, 'Datas diferentes cancelaram indevidamente.')

    def teste_cenario_clientes_diferentes():
        df_teste_raw = teste_criar_df([
            (1, 1, teste_data_1, 'C', 1, 'BRL', Decimal('900.00')),
            (2, 2, teste_data_1, 'D', 5, 'BRL', Decimal('900.00')),
        ])
        teste_saida = teste_aplicar_anulacao(df_teste_raw)
        teste_assert(teste_saida['teste_qt_pares'] == 0 and teste_saida['teste_qt_efetivo'] == 2, 'Clientes diferentes cancelaram indevidamente.')

    def teste_cenario_gate_residual_deliberado():
        df_teste_raw = teste_criar_df([
            (1, 1, teste_data_1, 'C', 1, 'BRL', Decimal('1000.00')),
            (2, 1, teste_data_1, 'D', 5, 'BRL', Decimal('1000.00')),
        ])
        df_teste_chaves = teste_obter_chaves(df_teste_raw)
        df_teste_invalido = df_teste_raw.select(*teste_colunas_funcionais)
        teste_capturou = False
        try:
            teste_exigir_sem_residuais(df_teste_invalido, df_teste_chaves)
        except AssertionError as teste_exc:
            teste_capturou = 'QT_CHAVES_COM_PARES_RESIDUAIS=' in str(teste_exc)
        teste_assert(teste_capturou, 'Gate residual deliberadamente inválido não foi bloqueado como esperado.')

    teste_registrar('ZERO_PARES_FAST_PATH', teste_cenario_zero_pares)
    teste_registrar('1C_1D', teste_cenario_1c1d)
    teste_registrar('2C_1D_DETERMINISMO', teste_cenario_2c1d)
    teste_registrar('1C_2D_DETERMINISMO', teste_cenario_1c2d)
    teste_registrar('MULTIPLOS_PARES', teste_cenario_multiplos_pares)
    teste_registrar('MOEDA_DIFERENTE', teste_cenario_moeda_diferente)
    teste_registrar('MOEDA_VAZIA', teste_cenario_moeda_vazia)
    teste_registrar('MOEDA_NULA', teste_cenario_moeda_nula)
    teste_registrar('VALOR_NULO', teste_cenario_valor_nulo)
    teste_registrar('DATAS_DIFERENTES', teste_cenario_datas_diferentes)
    teste_registrar('CLIENTES_DIFERENTES', teste_cenario_clientes_diferentes)
    teste_registrar('GATE_RESIDUAL_DELIBERADO', teste_cenario_gate_residual_deliberado)

    STATUS_TESTES_SINTETICOS = 'OK'
except Exception as teste_exc_geral:
    STATUS_TESTES_SINTETICOS = 'ERRO'
    remover_temp_view_resultado()
    print(f'[V23][TESTES] STATUS_TESTES_SINTETICOS={STATUS_TESTES_SINTETICOS}')
    print(f'[V23][TESTES] FALHA={type(teste_exc_geral).__name__}: {teste_exc_geral}')
    raise

teste_tempo_total = time.perf_counter() - teste_inicio
print('[V23][TESTES] CENARIO | STATUS')
for teste_nome, teste_status in teste_resultados:
    print(f'{teste_nome} | {teste_status}')
print(f'[V23][TESTES] TOTAL={len(teste_resultados)}; TEMPO_SEG={teste_tempo_total:.6f}')
print(f'[V23][GATE] STATUS_TESTES_SINTETICOS={STATUS_TESTES_SINTETICOS}')


In [ ]:
%%spark

inicio = time.perf_counter()
STATUS_VIEW_FINAL = 'NOT_CREATED'
HOMOLOGACAO_FUNCIONAL_V23 = 'N'

gate_erros = []
if STATUS_SCHEMA_Q3 not in ('OK', 'SKIPPED'):
    gate_erros.append(f'STATUS_SCHEMA_Q3={STATUS_SCHEMA_Q3}')
if STATUS_SCHEMA_Q5 not in ('OK', 'SKIPPED'):
    gate_erros.append(f'STATUS_SCHEMA_Q5={STATUS_SCHEMA_Q5}')
if JANELA_FINANCEIRA_DISPONIVEL:
    if STATUS_SCHEMA_Q5 != 'OK':
        gate_erros.append(f'Com janela, STATUS_SCHEMA_Q5 deve ser OK; obtido={STATUS_SCHEMA_Q5}')
    if STATUS_GATE_ANULACAO != 'OK':
        gate_erros.append(f'Com janela, STATUS_GATE_ANULACAO deve ser OK; obtido={STATUS_GATE_ANULACAO}')
else:
    if STATUS_SCHEMA_Q5 != 'SKIPPED':
        gate_erros.append(f'Sem janela, STATUS_SCHEMA_Q5 deve ser SKIPPED; obtido={STATUS_SCHEMA_Q5}')
    if STATUS_GATE_ANULACAO != 'SKIPPED':
        gate_erros.append(f'Sem janela, STATUS_GATE_ANULACAO deve ser SKIPPED; obtido={STATUS_GATE_ANULACAO}')
if STATUS_TESTES_SINTETICOS != 'OK':
    gate_erros.append(f'STATUS_TESTES_SINTETICOS={STATUS_TESTES_SINTETICOS}')
if STATUS_CONTRATO_FINAL != 'OK':
    gate_erros.append(f'STATUS_CONTRATO_FINAL={STATUS_CONTRATO_FINAL}')

if gate_erros:
    remover_temp_view_resultado()
    STATUS_VIEW_FINAL = 'BLOQUEADA'
    HOMOLOGACAO_FUNCIONAL_V23 = 'N'
    raise AssertionError('Publicação bloqueada pelos gates: ' + ' | '.join(gate_erros))

try:
    remover_temp_view_resultado()
    df_final.createOrReplaceTempView(VIEW_RESULTADO)
    temp_views_publicadas = obter_temp_views_resultado()
    if len(temp_views_publicadas) != 1:
        raise AssertionError(
            f'Esperada exatamente uma temporary view {VIEW_RESULTADO}; encontradas={len(temp_views_publicadas)}.'
        )
    if temp_views_publicadas[0].name != VIEW_RESULTADO or not bool(temp_views_publicadas[0].isTemporary):
        raise AssertionError('Objeto publicado não corresponde à temporary view contratual.')
    STATUS_VIEW_FINAL = 'OK'
    HOMOLOGACAO_FUNCIONAL_V23 = 'S'
except Exception:
    try:
        spark.catalog.dropTempView(VIEW_RESULTADO)
    except Exception:
        pass
    STATUS_VIEW_FINAL = 'ERRO'
    HOMOLOGACAO_FUNCIONAL_V23 = 'N'
    raise

medir_transformacao('VIEW_TEMPORARIA', inicio, 1, 80)
print('[RADAR_MVP] Contrato final e todos os gates aplicáveis foram aprovados.')
print(f'[RADAR_MVP] resultado disponível somente em {VIEW_RESULTADO}.')
print('[V23][STATUS FINAL]')
print(f'STATUS_SCHEMA_Q3 = {STATUS_SCHEMA_Q3}')
print(f'STATUS_SCHEMA_Q5 = {STATUS_SCHEMA_Q5}')
print(f'STATUS_GATE_ANULACAO = {STATUS_GATE_ANULACAO}')
print(f'STATUS_TESTES_SINTETICOS = {STATUS_TESTES_SINTETICOS}')
print(f'STATUS_CONTRATO_FINAL = {STATUS_CONTRATO_FINAL}')
print(f'STATUS_VIEW_FINAL = {STATUS_VIEW_FINAL}')
print(f'HOMOLOGACAO_FUNCIONAL_V23 = {HOMOLOGACAO_FUNCIONAL_V23}')
print(f'ESTRATEGIA_Q4 = {ESTRATEGIA_Q4}')
print(f'PERFORMANCE_Q4_HOMOLOGADA = {PERFORMANCE_Q4_HOMOLOGADA}')
df_final.show(1, truncate=False)


## Resumo consolidado de performance


In [ ]:
%%spark

tempo_total_mvp = time.perf_counter() - INICIO_MVP
metricas_ordenadas = sorted(METRICAS, key=lambda m: m['TEMPO_SEG'], reverse=True)

print('ETAPA / QUERY | FONTE | TEMPO | LINHAS | COLUNAS | STATUS')
for m in metricas_ordenadas:
    print(f"{m['ETAPA_QUERY']} | {m['FONTE']} | {m['TEMPO_SEG']:.6f}s | {m['LINHAS']} | {m['COLUNAS']} | {m['STATUS']}")

queries_registradas = [m for m in METRICAS if m['TIPO'] == 'QUERY']
queries_ok = [m for m in queries_registradas if m['STATUS'] == 'OK']
queries_skipped = [m for m in queries_registradas if m['STATUS'] == 'SKIPPED']
if len(queries_registradas) != 5:
    raise AssertionError(f'Devem existir exatamente cinco registros de consultas externas; encontrado={len(queries_registradas)}.')
transformacoes = [m for m in METRICAS if m['TIPO'] == 'TRANSFORMACAO']
tempo_queries = sum(m['TEMPO_SEG'] for m in queries_ok)
tempo_transformacoes = sum(m['TEMPO_SEG'] for m in transformacoes)
fonte_tempo = defaultdict(float)
for m in queries_ok:
    fonte_tempo[m['FONTE']] += m['TEMPO_SEG']

query_mais_lenta = max(queries_ok, key=lambda m: m['TEMPO_SEG']) if queries_ok else None
etapa_mais_lenta = max(transformacoes, key=lambda m: m['TEMPO_SEG']) if transformacoes else None
fonte_mais_lenta = max(fonte_tempo.items(), key=lambda item: item[1]) if fonte_tempo else None

print('=' * 80)
print(f'TEMPO TOTAL DO MVP: {tempo_total_mvp:.6f}s')
print(f'TEMPO TOTAL DAS QUERIES: {tempo_queries:.6f}s')
print(f'TEMPO TOTAL DAS TRANSFORMAÇÕES: {tempo_transformacoes:.6f}s')
print(f"QUERY MAIS LENTA: {query_mais_lenta['ETAPA_QUERY'] if query_mais_lenta else None}")
print(f"ETAPA MAIS LENTA: {etapa_mais_lenta['ETAPA_QUERY'] if etapa_mais_lenta else None}")
print(f'FONTE COM MAIOR TEMPO ACUMULADO: {fonte_mais_lenta}')
print(f'QUANTIDADE TOTAL DE QUERIES REGISTRADAS: {len(queries_registradas)}')
print(f'QUANTIDADE TOTAL DE QUERIES EXECUTADAS: {len(queries_ok)}')
print(f'QUANTIDADE TOTAL DE QUERIES SKIPPED: {len(queries_skipped)}')
print('=' * 80)
if len(queries_ok) > 5:
    raise AssertionError('Mais de cinco consultas externas foram executadas.')


## V23 — RESUMO CONSOLIDADO DE OBSERVABILIDADE

Separação entre o custo funcional do Radar e o custo adicional de auditoria do MVP.


In [ ]:
%%spark

tempo_total_notebook = time.perf_counter() - INICIO_MVP
metricas_v23 = list(METRICAS)
queries_v23 = [m for m in metricas_v23 if m['TIPO'] == 'QUERY']
queries_executadas_v23 = [m for m in queries_v23 if m['STATUS'] == 'OK']
queries_skipped_v23 = [m for m in queries_v23 if m['STATUS'] == 'SKIPPED']
transformacoes_v23 = [m for m in metricas_v23 if m['TIPO'] == 'TRANSFORMACAO']
auditorias_v23 = [m for m in metricas_v23 if m['TIPO'] == 'AUDITORIA']
validacoes_v23 = [m for m in metricas_v23 if m['TIPO'] == 'VALIDACAO']
tempo_queries_v23 = sum(m['TEMPO_SEG'] for m in queries_executadas_v23)
tempo_transformacoes_v23 = sum(m['TEMPO_SEG'] for m in transformacoes_v23)
tempo_auditorias_v23 = sum(m['TEMPO_SEG'] for m in auditorias_v23)
tempo_validacoes_v23 = sum(m['TEMPO_SEG'] for m in validacoes_v23)
tempo_pipeline_funcional_v23 = tempo_queries_v23 + tempo_transformacoes_v23 + tempo_validacoes_v23
fonte_tempo_v23 = defaultdict(float)
for metrica in queries_executadas_v23:
    fonte_tempo_v23[metrica['FONTE']] += metrica['TEMPO_SEG']
query_mais_lenta_v23 = max(queries_executadas_v23, key=lambda m: m['TEMPO_SEG']) if queries_executadas_v23 else None
transformacao_mais_lenta_v23 = max(transformacoes_v23, key=lambda m: m['TEMPO_SEG']) if transformacoes_v23 else None
auditoria_mais_lenta_v23 = max(auditorias_v23, key=lambda m: m['TEMPO_SEG']) if auditorias_v23 else None
fonte_mais_custosa_v23 = max(fonte_tempo_v23.items(), key=lambda item: item[1]) if fonte_tempo_v23 else None
print('=' * 80)
print(f'TEMPO_QUERIES = {tempo_queries_v23:.6f}s')
print(f'TEMPO_TRANSFORMACOES = {tempo_transformacoes_v23:.6f}s')
print(f'TEMPO_AUDITORIAS = {tempo_auditorias_v23:.6f}s')
print(f'TEMPO_VALIDACOES = {tempo_validacoes_v23:.6f}s')
print(f'TEMPO_PIPELINE_FUNCIONAL = {tempo_pipeline_funcional_v23:.6f}s')
print(f'TEMPO_TOTAL_NOTEBOOK = {tempo_total_notebook:.6f}s')
print(f'QUERIES_EXECUTADAS = {len(queries_executadas_v23)}')
print(f'QUERIES_SKIPPED = {len(queries_skipped_v23)}')
print(f"QUERY_MAIS_LENTA = {query_mais_lenta_v23['ETAPA_QUERY'] if query_mais_lenta_v23 else None}")
print(f"TRANSFORMACAO_MAIS_LENTA = {transformacao_mais_lenta_v23['ETAPA_QUERY'] if transformacao_mais_lenta_v23 else None}")
print(f"AUDITORIA_MAIS_LENTA = {auditoria_mais_lenta_v23['ETAPA_QUERY'] if auditoria_mais_lenta_v23 else None}")
print(f'FONTE_COM_MAIOR_TEMPO_ACUMULADO = {fonte_mais_custosa_v23}')
print('=' * 80)
if len(queries_v23) != 5:
    raise AssertionError(f'V23 deve preservar cinco consultas externas; encontrado={len(queries_v23)}.')

print('[V23][RESUMO DE GATES]')
print(f'STATUS_SCHEMA_Q3 = {STATUS_SCHEMA_Q3}')
print(f'STATUS_SCHEMA_Q5 = {STATUS_SCHEMA_Q5}')
print(f'STATUS_GATE_ANULACAO = {STATUS_GATE_ANULACAO}')
print(f'STATUS_TESTES_SINTETICOS = {STATUS_TESTES_SINTETICOS}')
print(f'STATUS_CONTRATO_FINAL = {STATUS_CONTRATO_FINAL}')
print(f'STATUS_VIEW_FINAL = {STATUS_VIEW_FINAL}')
print(f'HOMOLOGACAO_FUNCIONAL_V23 = {HOMOLOGACAO_FUNCIONAL_V23}')
print(f'PERFORMANCE_Q4_HOMOLOGADA = {PERFORMANCE_Q4_HOMOLOGADA}')


In [ ]:
%%spark

for nome_df in ['df_cliente_raw','df_ciclo_raw','df_renda_raw','df_perfil_raw','df_mov_raw','df_chaves_anulacao','df_mov_efetivo','df_classificado','df_brl']:
    obj = globals().get(nome_df)
    if obj is not None and hasattr(obj, 'unpersist'):
        obj.unpersist(blocking=False)
print('[RADAR_MVP] caches auxiliares liberados; a temporary view final homologada permanece disponível na sessão.')


## Encerramento

Resultado: uma linha, um CD_CLI, 80 colunas e schema contratual na view temporária vw_radar_financeiro_cliente_mvp.

Nenhuma tabela foi criada, alterada ou gravada.
